# EcoBin: A Two Stage Waste Classification Model


The United States alone generated over [292.4 million tons of Municipal Solid Waste in 2018](https://www.epa.gov/facts-and-figures-about-materials-waste-and-recycling/national-overview-facts-and-figures-materials). Material Recovery Facilities (MRFs) are responsible for sorting and processing recyclable waste so that it can be reused by manufacturers, reducing the amount of waste sent to landfills and limiting the environmental damage that follows. However, a startling **[1 in 4 items in our recycling stream are contaminated](https://recyclops.com/understanding-recycling-contamination/)** and cannot be processed by MRFs.

Recycling contamination stems from two sources:

1. **Wish-Cycling:** The practice of placing non-recyclable items like plastic bags and styrofoam into the recycling bin in the hope that they can be recycled. These items contaminate legitimate recyclables and drive up processing costs for MRFs. Fortunately, advances in *deep convolutional neural networks* have enabled classification models to categorize waste images with far greater accuracy than humans, reducing wish-cycling related problems.

2. **Misclassification:** Although machines are able to classify waste with astonishingly high accuracy, a critical flaw remains. All existing waste classification models are trained to classify items based on their material composition. This approach fails to account for contamination. Consider the example of a pizza-stained cardboard box. A waste classification model may determine that the pizza-stained cardboard box is ***materially recyclable*** becuase it is made out of cardboard; however it fails to account for the ***contamination*** of the grease and food residue on the box that make it suited for the garbage bin instead.



This notebook takes a two-stage approach to solving the waste classification problem. It trains a **Stage A: Base Waste Classifier** to categorize items into either garbage, curbside recycling, drop-off recycling or compost based on their material composition. It then trains a **Stage B: Contamination Classifier** to classify items as either a clean recyclable or a contaminated recyclable. Clean recyclables retain their Stage A classification, while contaminated recyclables are re-routed to garbage to reflect that they cannot be recycled in their current state.


## Step 0: Setup -- Imports, Paths and Hardware


Before we touch any data we will install the third-party packages we need, import every library used downstream, configure mixed precision and the dual T4 GPU strategy, set our random seed, and bind a constant to every file path the notebook writes to. Doing this once up front keeps every later section short -- nothing below this cell installs or imports anything new.

We will be training on **Kaggle GPU T4 x 2** with `mixed_float16` enabled, so each model gets ~2x the memory headroom and ~1.5x the throughput from the Tensor Cores. The final softmax (Stage A) and sigmoid (Stage B) layers are explicitly cast back to `float32` to keep the loss numerically stable.


In [ ]:
"""
This cell handles all setup for the notebook:

1. Installs the few packages that Kaggle's base TF 2.16 image does not ship.
2. Imports every library that any downstream section uses.
3. Enables mixed_float16 + tf.distribute.MirroredStrategy across both T4 GPUs.
4. Seeds Python, NumPy and TensorFlow for reproducibility.
5. Binds constants for every Kaggle input path and every /kaggle/working
   output directory the notebook writes to.
"""

# ---------- 1. Dependencies missing from the Kaggle TF image ----------
# Kaggle ships an ABI-matched numpy <-> scipy pair. Some rembg[gpu] transitive
# deps would otherwise upgrade numpy and silently break that pairing (sklearn
# import fails on a missing internal symbol). Pin both packages to their
# currently-installed versions via a constraint file so pip cannot touch them.
import numpy as _np, scipy as _sp
_nl = chr(10)
with open("/tmp/ecobin_pin.txt", "w") as _f:
    _f.write(f"numpy=={_np.__version__}{_nl}scipy=={_sp.__version__}{_nl}")
del _np, _sp, _nl
!pip install -q -c /tmp/ecobin_pin.txt "rembg[gpu]" pillow-avif-plugin statsmodels

# ---------- 2. Imports ----------
import os, gc, io, json, math, random, shutil, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image, ImageFilter
from tqdm import tqdm

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.optimizers.schedules import CosineDecay

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve,
)
from sklearn.model_selection import train_test_split

from rembg import remove, new_session
from scipy import ndimage

from statsmodels.stats.contingency_tables import mcnemar

# ---------- 3. Mixed precision + dual-GPU strategy ----------
tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
print(f"TF {tf.__version__} | Mixed precision : {tf.keras.mixed_precision.global_policy().name}")
print(f"GPUs visible              : {len(tf.config.list_physical_devices('GPU'))}")
print(f"MirroredStrategy replicas : {strategy.num_replicas_in_sync}")

# ---------- 4. Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- 5. Constants and paths ----------
IMG_SIZE   = 224
BATCH_SIZE = 32                       # Per-replica batch (global = BATCH_SIZE * num_replicas)
AUTOTUNE   = tf.data.AUTOTUNE
MAX_IMAGES = None                     # Set to an int to subsample Step 3 generation for smoke tests

# Kaggle input paths (datasets are pre-mounted as directories)
STAGE_A_DATASET = Path("/kaggle/input/datasets/alistairking/recyclable-and-household-waste-classification/images/images")
TEXTURE_PATH    = Path("/kaggle/input/datasets/ragbag84/waste-contamination-textures-dataset/Contamination Textures Dataset/textures")
TEST_SET_PATH   = Path("/kaggle/input/datasets/ragbag84/ecobin-pathway-test-set/Test_Dataset")

# /kaggle/working subdirectories that the notebook writes to
WORKING_PATH         = Path("/kaggle/working")
SPLITS_PATH          = WORKING_PATH / "splits"
MODELS_PATH          = WORKING_PATH / "Models"
DATASET_AUDIT_PATH   = WORKING_PATH / "Waste Classification Dataset Audit"
STAGE_A_RESULTS_PATH = WORKING_PATH / "Stage A Results"
STAGE_B_RESULTS_PATH = WORKING_PATH / "Stage B Results"
EVAL_RESULTS_PATH    = WORKING_PATH / "McNemar Evaluation Results"
STAGE_B_PATH         = WORKING_PATH / "Stage B Dataset"
TEXTURE_CACHE        = WORKING_PATH / "texture_cache"
DATASET_ROOT         = WORKING_PATH / "Synthetic Recyclable Contamination Dataset"

for _dir in [
    SPLITS_PATH, MODELS_PATH, DATASET_AUDIT_PATH,
    STAGE_A_RESULTS_PATH, STAGE_B_RESULTS_PATH, EVAL_RESULTS_PATH,
]:
    _dir.mkdir(parents=True, exist_ok=True)

# Step 3 contamination configuration
CONTAMINATION_LEVELS = {
    "light":  1,
    "medium": 2,
    "heavy":  3,
}


# Stage A class-weight configuration: historically confusable classes get
# extra loss weight so the optimizer invests more capacity in pulling them
# out of their confusion clusters.
HARD_CLASSES = [
    "aluminum_food_cans",
    "steel_food_cans",
    "plastic_soda_bottles",
    "paper_cups",
]
HARD_CLASS_WEIGHT = 1.75


# Stage A class-weight configuration: historically confusable classes get
# extra loss weight so the optimizer invests more capacity in pulling them
# out of their confusion clusters.
HARD_CLASSES = [
    "aluminum_food_cans",
    "steel_food_cans",
    "plastic_soda_bottles",
    "paper_cups",
]
HARD_CLASS_WEIGHT = 1.75

# OpenCV face cascade used by the Step 5 inference gate
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
assert not FACE_CASCADE.empty(), "Haar cascade failed to load -- check the cv2 install"

print("Setup complete.")

## Step 1: Prepare Waste Classification Dataset


### 1.1 Set the Dataset Path


We will be using the [Recyclable and Household Waste Classification Dataset](https://www.kaggle.com/datasets/alistairking/recyclable-and-household-waste-classification) by Alistair King to train the Stage A: Base Waste Classifier. This dataset contains 15,000 images (each 256x256 pixels) organized into 30 distinct classes of waste. We've chose this dataset becuase it covers a wide range of major waste categories like `Plastic`, `Paper and Cardboard`, `Glass`, `Metal`, `Organic Waste` and `Textiles`.

It is important to note that each class in this dataset is further divided into 2 subcategories: `default` (studio-like images of waste items) and `real_world` (images of waste items in real-world environments). This allows us to train the model on how waste looks like in the real-world.

Finally, each class in the dataset contains 500 images and the images are evenly divided between the two subcategories for each class. On Kaggle the dataset is pre-mounted as a directory at the path below, so we just bind a constant to it.


In [ ]:
DATASET_PATH = STAGE_A_DATASET
print(f"Dataset path : {DATASET_PATH}")
print(f"Classes found: {len(list(DATASET_PATH.iterdir()))}")


### 1.2 Map Each Class of Waste to a Disposal Pathway


Each class of waste must be disposed differently. We will map each class of waste within the dataset to one of four disposal pathways: **garbage**, **curbside recycling**, **drop-off recycling** and **compost**.

The United States has over 19,000 different municipalities and each municipality has different recycling rules and guidelines. Therefore, it's difficult to make a disposal mapping for each class of waste that's universally applicable to all municipalities in the nation. Consequently, we will be using the recycling guidelines for the City of Phoenix (the author's hometown) for this project.


In [ ]:
# Maps each class of waste within the dataset to a disposal pathway
DISPOSAL_MAP = {

    # Curbside Recycling: 17 classes
    'plastic_soda_bottles':       'curbside_recycling',
    'aerosol_cans':               'curbside_recycling',
    'steel_food_cans':            'curbside_recycling',
    'cardboard_boxes':            'curbside_recycling',
    'glass_beverage_bottles':     'curbside_recycling',
    'plastic_cup_lids':           'curbside_recycling',
    'cardboard_packaging':        'curbside_recycling',
    'glass_food_jars':            'curbside_recycling',
    'aluminum_food_cans':         'curbside_recycling',
    'plastic_food_containers':    'curbside_recycling',
    'magazines':                  'curbside_recycling',
    'aluminum_soda_cans':         'curbside_recycling',
    'plastic_detergent_bottles':  'curbside_recycling',
    'newspaper':                  'curbside_recycling',
    'office_paper':               'curbside_recycling',
    'plastic_water_bottles':      'curbside_recycling',
    'glass_cosmetic_containers':  'curbside_recycling',

    # Drop-off Recycling: 4 classes
    'plastic_shopping_bags':      'dropoff_recycling',
    'plastic_trash_bags':         'dropoff_recycling',
    'clothing':                   'dropoff_recycling',
    'shoes':                      'dropoff_recycling',

    # Compost: 4 classes
    'eggshells':                  'compost',
    'coffee_grounds':             'compost',
    'tea_bags':                   'compost',
    'food_waste':                 'compost',

    # Garbage: 5 classes
    'disposable_plastic_cutlery': 'garbage',
    'styrofoam_cups':             'garbage',
    'styrofoam_food_containers':  'garbage',
    'plastic_straws':             'garbage',
    'paper_cups':                 'garbage',
}

# Derived once here so Step 3 can reuse it
RECYCLABLE_CLASSES = [
    cls for cls, pathway in DISPOSAL_MAP.items()
    if pathway in ('curbside_recycling', 'dropoff_recycling')
]
print(f"Recyclable classes feeding Stage B: {len(RECYCLABLE_CLASSES)}")


### 1.3 Display Class Distribution


Now that we've mapped each class of waste to a disposal route, we can now audit the dataset. This cell looks up each class of waste's assigned disposal pathway from `DISPOSAL_MAP`, counts the number of images in that class and calculates the percent composition of each class out of the total size of the dataset. It then displays the metrics for each class in a table and writes a csv to `/kaggle/working`.


In [ ]:
#---------------
# Table View
# --------------

# Appends each class of waste with disposal pathway and number of images into an empty array named records
records = []
for class_dir in sorted(DATASET_PATH.iterdir()):
    if not class_dir.is_dir():
        continue
    class_name = class_dir.name
    count = sum(1 for f in class_dir.rglob('*') if f.is_file())
    pathway = DISPOSAL_MAP.get(class_name, 'UNMAPPED')
    records.append({'class': class_name, 'count': count, 'pathway': pathway})

# Sorts values in the dataframe by count
df = pd.DataFrame(records).sort_values('count', ascending=False)

# Calculate percent composition of each class out of total dataset size
df['pct'] = (df['count'] / df['count'].sum() * 100).round(2)

# Print table and write csv to the audit folder
print(df[['class', 'count', 'pct', 'pathway']].to_string(index=False))
table_path = DATASET_AUDIT_PATH / 'class_distribution_table.csv'
df[['class', 'count', 'pct', 'pathway']].to_csv(table_path, index=False)
print(f"\nSaved to {table_path}")


In [ ]:
#---------------
# Bar Chart View
# --------------

# Maps each disposal pathway to its legend color
PATHWAY_COLORS = {
    'curbside_recycling': '#1565C0',
    'dropoff_recycling':  '#6A1B9A',
    'compost':            '#F57F17',
    'garbage':            '#2E7D32',
    'UNMAPPED':           '#B71C1C',
}

fig, ax = plt.subplots(figsize=(20, 6))
ax.bar(df['class'], df['count'], width=0.5,
       color=[PATHWAY_COLORS.get(p, '#B71C1C') for p in df['pathway']],
       edgecolor='black', linewidth=0.8)

ax.set_xticks(range(len(df['class'])))
ax.set_xticklabels(df['class'], rotation=45, ha='right', fontsize=9)
ax.tick_params(axis='x', pad=10)

ax.set_xlabel('Classes of Waste', fontweight='bold', fontsize=14, labelpad=15)
ax.set_ylabel('Image Count', fontweight='bold', fontsize=14)
ax.set_title('Recyclable and Household Waste Dataset Class Distribution by Disposal Pathway',
             fontweight='bold', fontsize=16)

legend = ax.legend(
    handles=[mpatches.Patch(color=c, label=p.replace('_', ' ').title())
             for p, c in PATHWAY_COLORS.items() if p in df['pathway'].values],
    loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0,
    title='Legend', title_fontproperties={'weight': 'bold', 'size': 10},
    frameon=True, edgecolor='black', fancybox=False,
)
legend.get_frame().set_linewidth(2)

plt.tight_layout()
plt.savefig(DATASET_AUDIT_PATH / 'class_distribution_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()


In this case we can see in both the table and the bar chart that all classes in the dataset have the same sample size (e.g. 500 images), so we don't have to worry about assigning class weights.


### 1.4 Split Dataset into Train/Val/Test


To train the model we will split the dataset into three paths: **train (70%)**, **validation (15%)** and **test (15%)**. During the split, test images are additionally written into two subcategory-specific directories, `test_default` and `test_real_world`, based on the `default` or `real_world` subfolder each image comes from. These directories let us evaluate the model on separate test sets in section 2.8.


In [ ]:
# Clear old splits if they exist so we always start from a clean state
if SPLITS_PATH.exists():
    shutil.rmtree(SPLITS_PATH)
    print("Cleared old splits")
SPLITS_PATH.mkdir(parents=True, exist_ok=True)

# Loop through all images in each class and assign to train, val, or test
for class_dir in sorted(DATASET_PATH.iterdir()):
    if not class_dir.is_dir():
        continue
    class_name = class_dir.name

    images = [f for f in class_dir.rglob('*') if f.is_file()]
    if len(images) < 10:
        print(f"WARNING: {class_name} only has {len(images)} images - skipping")
        continue

    train_imgs, temp    = train_test_split(images, test_size=0.30, random_state=SEED)
    val_imgs, test_imgs = train_test_split(temp,   test_size=0.50, random_state=SEED)

    for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
        split_dir = SPLITS_PATH / split / class_name
        split_dir.mkdir(parents=True, exist_ok=True)
        for img in imgs:
            # Prefix with parent subfolder name to avoid filename collisions
            unique_name = f"{img.parent.name}_{img.name}"
            shutil.copy2(img, split_dir / unique_name)

            # Copy test images into subcategory directories for Section 2.8 evaluation
            if split == 'test':
                subcat_dir = SPLITS_PATH / f'test_{img.parent.name}' / class_name
                subcat_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(img, subcat_dir / unique_name)

print("Train/val/test split complete")
print(f"  Train:           {sum(1 for _ in (SPLITS_PATH/'train').rglob('*')           if _.is_file())} images")
print(f"  Val:             {sum(1 for _ in (SPLITS_PATH/'val').rglob('*')             if _.is_file())} images")
print(f"  Test:            {sum(1 for _ in (SPLITS_PATH/'test').rglob('*')            if _.is_file())} images")
print(f"  Test Default:    {sum(1 for _ in (SPLITS_PATH/'test_default').rglob('*')    if _.is_file())} images")
print(f"  Test Real World: {sum(1 for _ in (SPLITS_PATH/'test_real_world').rglob('*') if _.is_file())} images")


## Step 2: Train the Stage A: Base Waste Classifier


### 2.1 Load Split Dataset


We begin by loading the train, validation, and test splits from the disk onto the TensorFlow datasets. All images are then resized to 224x224 pixels and grouped into batches of 32. Two additional datasets, `test_default_ds` and `test_real_world_ds`, are loaded from the subcategory directories created in Step 1.4.

The data pipeline is optimized for GPU training using three operations:

- **cache:** stores images in memory after the first epoch so they are not re-read from disk each epoch
- **shuffle:** randomly reorders the training set each epoch so the model does not see classes in the same order
- **prefetch:** loads the next batch on the CPU while the GPU trains on the current one, eliminating idle time between batches

**Note:** Shuffling is applied to the training set only to prevent the model from learning patterns in the order of the data. The order the images appear in the validation and test sets has no effect on the final metrics, so images are not shuffled in those sets. Only the train and validation sets are cached -- the three test sets are iterated once at evaluation time, so holding them in RAM would just waste ~3 GB that Step 4 needs for the Stage B dataset.


In [ ]:
"""
Load split datasets from disk into TensorFlow datasets.
- label_mode='categorical' returns one-hot labels (required by CategoricalCrossentropy below)
- train + val are cached in memory; test sets are not (iterated once each, ~3 GB saved)
"""

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLITS_PATH / 'train'),
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    seed=SEED, label_mode='categorical',
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLITS_PATH / 'val'),
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    seed=SEED, label_mode='categorical',
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLITS_PATH / 'test'),
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    seed=SEED, label_mode='categorical',
)
test_default_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLITS_PATH / 'test_default'),
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    seed=SEED, label_mode='categorical',
)
test_real_world_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLITS_PATH / 'test_real_world'),
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    seed=SEED, label_mode='categorical',
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {class_names}")

# Cache + prefetch only what is iterated more than once
train_ds           = train_ds.cache().shuffle(2000, seed=SEED).prefetch(AUTOTUNE)
val_ds             = val_ds.cache().prefetch(AUTOTUNE)
test_ds            = test_ds.prefetch(AUTOTUNE)
test_default_ds    = test_default_ds.prefetch(AUTOTUNE)
test_real_world_ds = test_real_world_ds.prefetch(AUTOTUNE)


### 2.2 Apply Data Augmentation Techniques


We apply data augmentation techniques to the train set to prevent the model from overfitting. ***Model Overfitting*** occurs when the model learns the training data too well capturing and doesn't learn the underlying patterns. This usually results in high accuracy on training data but low accuracy on validation and test splits. To prevent overfitting for our Stage A: Base Waste Classifier we manipulate and distort the images using data augmentation to ensure the model learns underlying patterns of the training data.

| Technique | Parameter | Purpose |
|---|---|---|
| RandomRotation | Ã‚Â±60Ã‚Â° | Randomly rotates training images by 60 degrees left or right |
| RandomZoom | Ã‚Â±20% | Randomly zooms in or out on the training image by 20% |
| RandomTranslation | Ã‚Â±15% | Randomly translates training images 15% left or right |
| RandomFlip | Horizontal and vertical | Randomly flips the training image horizontally or vertically |
| RandomBrightness | Ã‚Â±30% | Randomly increases or decreases lighting on training image by 30% |
| RandomContrast | Ã‚Â±30% | Randomly increases or decreases contrast on training image by 30% |

**Note:** Augmentation is applied during training becuase we're preventing the model from overfitting to the data in that phase. The validation and test sets will only use unaugmented images.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(factor=60/360),
    layers.RandomZoom(height_factor=0.2),
    layers.RandomTranslation(height_factor=0.15, width_factor=0.15),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomBrightness(factor=0.3),
    layers.RandomContrast(factor=0.3),
], name="augmentation")


### 2.3 Build the Stage A Base Waste Classifier


For the Stage A Base Waste Classifier we will be using a MobileNetV3Large model as a frozen feature extractor with a custom classification head. We use transfer learning from ImageNet weights, so we don't have to train the model on low-level feature detection from scratch. Instead we focus on only training the classification head (the last layer) across the 30 classes of waste. The model is built inside `strategy.scope()` so the variables are mirrored across both T4 GPUs, and the final softmax is cast back to `float32` so mixed precision does not destabilize the loss.

| Layer | Output Shape | Params | Purpose |
|---|---|---|---|
| InputLayer | (None, 224, 224, 3) | 0 | Defines the expected image dimensions to be 224Ãƒâ€”224 pixels across 3 RGB channels |
| Data Augmentation | (None, 224, 224, 3) | 0 | Applies random transformations from Step 2.2 during training only |
| MobileNetV3Large | (None, 7, 7, 960) | 2,996,352 | Pretrained feature extractor with ImageNet weights compresses each image into a 7Ãƒâ€”7 grid of 960 features |
| GlobalAveragePooling2D | (None, 960) | 0 | Collapses the 7Ãƒâ€”7Ãƒâ€”960 feature map into a flat 960-value vector by averaging each channel |
| BatchNormalization | (None, 960) | 3,840 | Normalizes the pooled feature vector to stabilize training and improve gradient flow |
| Dropout | (None, 960) | 0 | Randomly zeros 40% of features during training |
| Dense (256, ReLU + L2) | (None, 256) | 246,016 | Learns a compressed 256-dimensional representation with L2 regularization (1e-4) to penalize large weights and improve generalization |
| Dense (30, Softmax, fp32) | (None, 30) | 7,710 | Outputs a probability distribution across all 30 waste classes, cast to float32 |


In [ ]:
def build_stage_a(num_classes):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    base_model = EfficientNetV2S(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        include_preprocessing=True,
    )
    # Rename after construction -- EfficientNetV2S uses name internally as a
    # block-args dict key so it cannot be customised via the constructor.
    base_model._name = 'stage_a_backbone'  # Stage B looks this up with get_layer()
    base_model.trainable = False
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    # Final softmax in fp32 for numerical stability under mixed_float16
    outputs = layers.Dense(num_classes, activation='softmax',
                           dtype='float32', name='predictions')(x)
    return Model(inputs, outputs, name="stage_a"), base_model

# Build under the MirroredStrategy scope so weights are mirrored across both GPUs
with strategy.scope():
    model_a, base_model = build_stage_a(NUM_CLASSES)
model_a.summary()

### 2.4 Define Training Callbacks


Callbacks are objects that monitor training and automatically take actions at the end of each epoch. In this cell we define three callbacks and pass them into the `model.fit()` call in Step 2.5.

| Callback | Monitors | Action |
|---|---|---|
| ModelCheckpoint | val_accuracy | Saves model weights to disk whenever val_accuracy improves |
| EarlyStopping | val_accuracy | Stops training if val_accuracy does not improve for 7 consecutive epochs |
| ReduceLROnPlateau | val_loss | Halves the learning rate if val_loss does not improve for 3 consecutive epochs, down to a floor of 1e-6 |


In [ ]:
# Save best weights whenever val_accuracy improves
checkpoint_cb = ModelCheckpoint(
    str(MODELS_PATH / 'stage_a_best.keras'),
    monitor='val_accuracy', save_best_only=True, verbose=1,
)

# Stop training if val_accuracy plateaus for 7 epochs
early_stop_cb = EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True,
)

# Halve learning rate when val_loss stalls, down to 1e-6
reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1,
)


### 2.5 Stage A Model Training


This cell trains the Stage A: Base Waste Classifier across two phases.

**Phase 1 -- Head warmup (15 epochs, lr=1e-3, backbone fully frozen).** Only the new classification head trains. This lets the head reach a sensible basin before any backbone weights are touched, so the gradients in Phase 2 do not destroy the pre-trained ImageNet features.

**Phase 2 -- Fine-tune top 35% of backbone (25 epochs, lr=1e-5 with cosine decay).** The bottom 65% of EfficientNetV2-S stays frozen so the early generic edge/colour filters remain untouched. `BatchNormalization` layers are held in inference mode throughout Phase 2 -- their ImageNet running statistics should not drift on the smaller waste dataset. The learning rate decays smoothly from 1e-5 to 0 across the 25 epochs.

A class-weight dictionary is passed into `model.fit`: the four historically confusable classes in `HARD_CLASSES` receive 1.75x loss weight so the optimizer pushes harder on them. Phase 1 uses `ReduceLROnPlateau` (no early stop, since the head has not had time to converge); Phase 2 uses `EarlyStopping` instead (since `CosineDecay` already manages the learning rate, plateau-based reduction would conflict).

In [ ]:
def combine_histories(*histories):
    """Concatenate per-metric lists from multiple Keras History objects into one dict."""
    merged = {}
    for h in histories:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

# Class weights: hard classes receive HARD_CLASS_WEIGHT, everything else 1.0
class_weights_a = {i: 1.0 for i in range(NUM_CLASSES)}
for cls in HARD_CLASSES:
    if cls in class_names:
        class_weights_a[class_names.index(cls)] = HARD_CLASS_WEIGHT

# ---------- Phase 1: head warmup, backbone fully frozen ----------
print("=== Stage A Phase 1: head warmup (backbone frozen) ===")
base_model.trainable = False
with strategy.scope():
    model_a.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy'],
    )
history_a_p1 = model_a.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[checkpoint_cb, reduce_lr_cb],
    class_weight=class_weights_a,
    verbose=1,
)

# ---------- Phase 2: fine-tune top 35% of backbone with cosine-decay LR ----------
print("\n=== Stage A Phase 2: fine-tune top 35% backbone (cosine decay) ===")
n_layers      = len(base_model.layers)
unfreeze_from = int(n_layers * 0.65)
for i, layer in enumerate(base_model.layers):
    if i < unfreeze_from or isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

steps_per_epoch = int(train_ds.cardinality())
schedule        = CosineDecay(initial_learning_rate=1e-5, decay_steps=25 * steps_per_epoch)
with strategy.scope():
    model_a.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=schedule),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy'],
    )
history_a_p2 = model_a.fit(
    train_ds,
    epochs=25,
    validation_data=val_ds,
    callbacks=[checkpoint_cb, early_stop_cb],
    class_weight=class_weights_a,
    verbose=1,
)

# Stitch the two phase histories together for the plotting cell
history_a = type('H', (), {})()
history_a.history = combine_histories(history_a_p1, history_a_p2)

print(f"\nStage A model saved to: {MODELS_PATH / 'stage_a_best.keras'}")

### 2.6 Plot Stage A Training vs Validation Error Metrics


Before we evaluate the Stage A: Base Waste Classifier on the held-out test set, we have to examine its training/validation accuracy and loss curves. A healthy run shows training accuracy climbing steadily while validation accuracy tracks closely behind it, with both training and validation loss falling and remaining close together. A tight gap between the train and val curves indicates the model is generalizing well rather than memorizing the training data, which is what we want as it signals there isn't any overfitting.


In [ ]:
epochs = range(1, len(history_a.history['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
ax1.plot(epochs, history_a.history['accuracy'],     label='Train Accuracy', color='steelblue')
ax1.plot(epochs, history_a.history['val_accuracy'], label='Val Accuracy',   color='tomato')
ax1.set_title('Stage A Training vs Validation Accuracy')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Plot loss
ax2.plot(epochs, history_a.history['loss'],     label='Train Loss', color='steelblue')
ax2.plot(epochs, history_a.history['val_loss'], label='Val Loss',   color='tomato')
ax2.set_title('Stage A Training vs Validation Loss')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Stage A Base Waste Classifier Error Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_error_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.7 Evaluate Stage A Base Waste Classifier on Test Set


Now we can evaluate the Stage A: Base Waste Classifier on the held out test set. The test set has a total of 2,250 images, meaning each class will have 75 test images. We will be analyzing 4 metrics when evaluating the results of the test set.

1. **Accuracy:** The proportion of all predictions that are correct.
2. **Precision:** The measure of how often the model is correct when it predicts a given class.
3. **Recall:** The measure of how often the model correctly identifies all true members of a class.
4. **F1:** The harmonic mean of precision and recall.

Together, these four metrics give a more complete picture of model behavior than any single number alone. A model could achieve high accuracy by performing well on easy, common classes while failing on harder ones. Examining all four metrics per class reveals exactly where the model is reliable and where it is not.


In [ ]:
# ---------------
# Table View
# ---------------

best_model_a = tf.keras.models.load_model(MODELS_PATH / 'stage_a_best.keras')

# Run predictions on full test set
all_preds, all_labels = [], []
for images, labels in test_ds:
    preds = best_model_a.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute per-class accuracy and sort descending
cm = confusion_matrix(all_labels, all_preds)
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
sorted_indices     = np.argsort(per_class_accuracy)[::-1]

report = classification_report(
    all_labels, all_preds, target_names=class_names, digits=4, output_dict=True,
)

print("Stage A Classification Report")
print("=" * 75)
print(f"{'Class':<35} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
print("-" * 75)
for i in sorted_indices:
    name = class_names[i]
    print(f"{name:<35} {per_class_accuracy[i]:>9.4f} {report[name]['precision']:>10.4f} "
          f"{report[name]['recall']:>8.4f} {report[name]['f1-score']:>8.4f} {int(report[name]['support']):>9}")
print("-" * 75)
print(f"{'macro avg':<35} {np.mean(per_class_accuracy):>9.4f} "
      f"{report['macro avg']['precision']:>10.4f} {report['macro avg']['recall']:>8.4f} "
      f"{report['macro avg']['f1-score']:>8.4f} {int(report['macro avg']['support']):>9}")


In [ ]:
# ---------------
# Bar Chart View
# ---------------

# Per-class metrics
per_class_precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
per_class_recall    = recall_score(all_labels, all_preds, average=None, zero_division=0)
per_class_f1        = f1_score(all_labels, all_preds, average=None, zero_division=0)

# Macro averages
avg_accuracy  = np.mean(per_class_accuracy)
avg_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
avg_recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
avg_f1        = f1_score(all_labels, all_preds, average='macro', zero_division=0)

# Sort by descending accuracy with the Average bar first
sorted_labels    = ['Average'] + [class_names[i] for i in sorted_indices]
sorted_accuracy  = [avg_accuracy]  + [per_class_accuracy[i]  for i in sorted_indices]
sorted_precision = [avg_precision] + [per_class_precision[i] for i in sorted_indices]
sorted_recall    = [avg_recall]    + [per_class_recall[i]    for i in sorted_indices]
sorted_f1        = [avg_f1]        + [per_class_f1[i]        for i in sorted_indices]

x     = np.arange(len(sorted_labels))
width = 0.2

fig, ax = plt.subplots(figsize=(24, 7))
ax.bar(x - 1.5*width, sorted_accuracy,  width, label='Accuracy',  color='steelblue',   edgecolor='black', linewidth=0.8)
ax.bar(x - 0.5*width, sorted_precision, width, label='Precision', color='tomato',       edgecolor='black', linewidth=0.8)
ax.bar(x + 0.5*width, sorted_recall,    width, label='Recall',    color='seagreen',     edgecolor='black', linewidth=0.8)
ax.bar(x + 1.5*width, sorted_f1,        width, label='F1 Score',  color='mediumpurple', edgecolor='black', linewidth=0.8)

ax.set_title('Stage A Base Waste Classifier Per-Class Metrics', fontsize=16, fontweight='bold')
ax.set_xlabel('Class of Waste', fontsize=14)
ax.set_ylabel('Score', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(sorted_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOverall Metrics")
print(f"{'='*40}")
print(f"Accuracy:  {avg_accuracy:.4f}")
print(f"Precision: {avg_precision:.4f}")
print(f"Recall:    {avg_recall:.4f}")
print(f"F1 Score:  {avg_f1:.4f}")


After reading the table and visualizing the results on the bar chart, we can make a couple of conlusions.

**Overall Performance:** Stage A achieves a macro-average accuracy of **XX.XX%**, with precision, recall, and F1 all clustering tightly around the same value. The consistency across these three metrics suggests the model is neither systematically over-predicting nor under-predicting any particular disposal pathway at the aggregate level, which is good as it tells us our training strategy was effective and reliable.

However, the accuracy of around **XX%** is low compared to state-of-the-art waste classification models which often boast accuracy rates of 95% or greater. However, we can excuse this as we are training the model across 30 different outputs and our goal is accuracy in disposal routing (garbage, curbside recycling, drop-off recycling and compost), not what the object actually is.

**Strong performers:** The top of the table is dominated by visually distinctive classes (fill in the top 3 with their per-class accuracies after the run). These classes tend to have consistent textures and colors that make them relatively easy to discriminate even across the different image sources in the dataset.

**Weak performers:** The bottom of the table reveals a clear pattern (fill in the bottom 3 with their per-class accuracies after the run). The cardboard confusion is unsurprising: cardboard boxes and cardboard packaging are visually nearly identical and share the same material, differing only in form factor. The model struggles to learn a reliable boundary between them. Similarly, aluminum food cans and steel food cans are easily confused due to their similar cylindrical shape and metallic appearance.

**Precision-recall asymmetry:** Several classes show a notable gap between precision and recall. Discuss any class with a precision/recall gap of more than 10 percentage points after the run.


### 2.8 Default vs Real World Test Set Evaluation


Up to this point, Stage A has been evaluated on a held-out test set drawn from the full dataset, which mixes both the `default` and `real_world` images together. That overall accuracy number is useful, but it obscures an important question: does Stage A perform equally well on studio images and real-world images, or does performance degrade when the background is cluttered and lighting is inconsistent?

In this cell we'll compare the results of the Stage A: Base Waste Classifier on both the `default` and `real_world` test sets. The difference between in accuracy is the domain gap. Generally, we expect the model to have higher accuracy on the `default` images.


In [ ]:
def evaluate_subcategory(model, dataset):
    preds, labels = [], []
    for images, label_batch in dataset:
        batch_preds = model.predict(images, verbose=0)
        preds.extend(np.argmax(batch_preds, axis=1))
        labels.extend(np.argmax(label_batch.numpy(), axis=1))
    return np.array(preds), np.array(labels)

preds_default, labels_default = evaluate_subcategory(best_model_a, test_default_ds)
preds_real,    labels_real    = evaluate_subcategory(best_model_a, test_real_world_ds)

def per_class_acc(labels, preds):
    cm_sub  = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    row_sum = cm_sub.sum(axis=1)
    row_sum[row_sum == 0] = 1   # Avoid divide-by-zero for empty classes
    return cm_sub.diagonal() / row_sum

acc_default = per_class_acc(labels_default, preds_default)
acc_real    = per_class_acc(labels_real,    preds_real)

print(f"Default    Overall Accuracy: {acc_default.mean():.4f}")
print(f"Real World Overall Accuracy: {acc_real.mean():.4f}")
print(f"Domain Gap:                 {acc_default.mean() - acc_real.mean():+.4f}")

sorted_by_real = np.argsort(acc_real)
print(f"\n{'Class':<35} {'Default':>9} {'Real World':>11} {'Gap':>8}")
print("-" * 66)
for i in sorted_by_real:
    gap = acc_default[i] - acc_real[i]
    print(f"{class_names[i]:<35} {acc_default[i]:>9.4f} {acc_real[i]:>11.4f} {gap:>+8.4f}")

# Plot side-by-side bars
sort_idx       = np.argsort(acc_real)
sorted_names   = [class_names[i] for i in sort_idx]
sorted_default = [acc_default[i] * 100 for i in sort_idx]
sorted_real    = [acc_real[i]    * 100 for i in sort_idx]

y      = np.arange(len(sorted_names))
height = 0.35

fig, ax = plt.subplots(figsize=(14, 12))
ax.barh(y - height/2, sorted_default, height, label='Default',    color='steelblue', edgecolor='black', linewidth=0.5)
ax.barh(y + height/2, sorted_real,    height, label='Real World', color='tomato',    edgecolor='black', linewidth=0.5)

ax.set_yticks(y)
ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('Stage A Per-Class Accuracy: Default vs Real World', fontweight='bold', fontsize=13)
ax.set_xlim(0, 110)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_default_vs_real_world.png', dpi=150, bbox_inches='tight')
plt.show()


Stage A achieves a **default subcategory accuracy of XX.X%** and a **real-world subcategory accuracy of XX.X%**, producing an overall domain gap of **X.X** percentage points. Discuss whether this gap is wider or narrower than expected after the run.


### 2.9 Confusion Matrices


We'll plot two confusion matrices for the Stage A: Base Waste Classifer. The first is a **class-level confusion matrix across all 30 waste categories**. Confusion matrices provide a detailed breakdown of model performance by comparing each predicted class against the actual ground truth label. A well-performing model will show a dark diagonal from top-left to bottom-right, indicating that most items are being classified correctly.

The second is a **disposal pathway confusion matrix collapsed across the 4 disposal pathways**. This is the more meaningful evaluation for EcoBin's real-world purpose. What ultimately matters is not whether the model identifies the exact object, but whether it routes that object to the correct bin. Evaluating at the pathway level also naturally absorbs errors between visually similar classes that share the same route, such as `cardboard_boxes` and `cardboard_packaging`, which both map to curbside recycling. A class-level error between these two is inconsequential in practice and should not count against the model.


In [ ]:
# ----------------------------
# Class-level Confusion Matrix
# ----------------------------

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names, ax=ax,
    linewidths=0.5, annot_kws={'size': 6}, vmin=0, vmax=1,
)
ax.set_title('Stage A Base Waste Classifier Confusion Matrix (30 Classes)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=14)
ax.set_ylabel('True Label',      fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0,             fontsize=9)
plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_confusion_matrix_classes.png', dpi=150, bbox_inches='tight')
plt.show()

# ---------------------------------
# Disposal Pathway Confusion Matrix
# ---------------------------------

PATHWAY_ORDER  = ['curbside_recycling', 'dropoff_recycling', 'compost', 'garbage']
PATHWAY_LABELS = [p.replace('_', ' ').title() for p in PATHWAY_ORDER]

pathway_true = [DISPOSAL_MAP.get(class_names[i], 'UNMAPPED') for i in all_labels]
pathway_pred = [DISPOSAL_MAP.get(class_names[i], 'UNMAPPED') for i in all_preds]

cm_pathway      = confusion_matrix(pathway_true, pathway_pred, labels=PATHWAY_ORDER)
cm_pathway_norm = cm_pathway.astype(float) / cm_pathway.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_pathway_norm, annot=True, fmt='.2f', cmap='Greens',
    xticklabels=PATHWAY_LABELS, yticklabels=PATHWAY_LABELS, ax=ax,
    linewidths=1, annot_kws={'size': 12}, vmin=0, vmax=1,
)
ax.set_title('Stage A Base Waste Classifier Confusion Matrix (Disposal Pathways)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Pathway', fontsize=14)
ax.set_ylabel('True Pathway',      fontsize=14)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(rotation=0,             fontsize=10)
plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_confusion_matrix_pathways.png', dpi=150, bbox_inches='tight')
plt.show()


We can see that accuracy has jumped significantly when we collapse from 30 class labels down to the 4 disposal pathways. This is expected: errors between visually similar classes that share a pathway (e.g. `cardboard_boxes` predicted as `cardboard_packaging`) no longer count as misclassifications. Next we will plot a bar chart that quantifies this jump class-by-class.


### 2.10 Plot Adjusted Accuracy Bar Chart


After comparing the confusion matrices in the previous cell, we can determine that class-level accuracy penalizes the model equally for all misclassifications, regardless of whether the error actually results in an item being sent to the wrong bin.

Adjusted accuracy corrects for this by marking a prediction as correct whenever the predicted class shares the same disposal pathway as the true class, regardless of whether the exact class label matches. In other words adjusted accuracy doesn't mark the model as incorrect if it classifies `cardboard_box` as `cardboard_packaging` becuase both items go to curbside recycling either way. This adjusted approach reflects only the genuine mistakes that would send an item to the wrong disposal pathway.


In [ ]:
# Adjusted accuracy: correct if predicted class shares a disposal pathway with the true class
adjusted_correct = np.zeros(NUM_CLASSES)
class_counts     = np.zeros(NUM_CLASSES)

for true_idx, pred_idx in zip(all_labels, all_preds):
    true_pathway = DISPOSAL_MAP.get(class_names[true_idx], 'UNMAPPED')
    pred_pathway = DISPOSAL_MAP.get(class_names[pred_idx], 'UNMAPPED')
    class_counts[true_idx] += 1
    if true_pathway == pred_pathway:
        adjusted_correct[true_idx] += 1

adj_acc_per_class = adjusted_correct / class_counts

# Sort by original accuracy ascending so highest sits at top
sort_idx     = np.argsort(per_class_accuracy)
sorted_names = [class_names[i] for i in sort_idx]
sorted_orig  = [per_class_accuracy[i] * 100 for i in sort_idx]
sorted_adj   = [adj_acc_per_class[i]  * 100 for i in sort_idx]

y = np.arange(len(sorted_names))

fig, ax = plt.subplots(figsize=(12, 12))
ax.barh(y, sorted_adj,  color='#A8D5A2', label='Adjusted Accuracy', edgecolor='black', linewidth=0.5)
ax.barh(y, sorted_orig, color='#2E7D32', label='Original Accuracy',  edgecolor='black', linewidth=0.5)

ax.set_yticks(y); ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('Per-Class Accuracy: Original vs Adjusted (City of Phoenix Policy)', fontweight='bold', fontsize=13)
ax.set_xlim(0, 110); ax.grid(axis='x', alpha=0.3)
legend = ax.legend(loc='lower right', frameon=True, edgecolor='black', fancybox=False, fontsize=10)
legend.get_frame().set_linewidth(1.5)

plt.tight_layout()
plt.savefig(STAGE_A_RESULTS_PATH / 'stage_a_adjusted_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

overall_orig = per_class_accuracy.mean()
overall_adj  = adjusted_correct.sum() / class_counts.sum()
print(f"Original Accuracy: {overall_orig:.4f}")
print(f"Adjusted Accuracy: {overall_adj:.4f}")
print(f"Pathway Gain:      {overall_adj - overall_orig:+.4f}")


The bar chart shows that collapsing to disposal pathways lifts accuracy by **+XX percentage points**, bringing our deployment-relevant accuracy to **XX.XX%**.


## Step 3: Prepare Synthetic Recyclable Contamination Dataset

Stage B requires a training dataset that the AlistairKing dataset does not provide: recyclable images labelled as either clean or contaminated. We build this dataset entirely from the recyclable images already in the Stage A source data, without collecting any new images.

We record each source image as-is and label it **clean**. We then generate one contaminated copy per source image at a severity level (**light**, **medium** or **heavy**) that is pre-assigned before generation begins. Severity levels are distributed evenly across the full image pool so each level accounts for roughly one third of the contaminated set. This produces a final dataset of **~10,500 clean images and ~10,500 contaminated images, balanced 1:1**.

We apply contamination by compositing texture patches from a separate contamination texture library onto each object. A U2-Net segmentation model isolates the foreground object so that no contamination bleeds onto the background.


### 3.1 Initialise the rembg Session

Without a session argument, `rembg` reloads the U2-Net ONNX model from disk every time `remove()` is called. With over 10,000 images to generate this adds up to thousands of redundant model loads. We create one session here and reuse it throughout the pipeline to eliminate that overhead. With `rembg[gpu]` installed and a GPU accelerator enabled, ONNX Runtime routes inference through CUDA automatically.


In [ ]:
REMBG_SESSION = new_session('u2netp')
print("U2-Net session initialised")


### 3.2 Build the Contamination Texture Pool


#### 3.2.1 Discover Contaminant Types and Texture Images

We build the texture pool by scanning the contamination texture library directory. Each subdirectory represents one contaminant type and contains five source PNG texture images. The pool is a dictionary mapping each contaminant type name to the list of file paths for its textures.

This cell prints each contaminant type and the number of images within each contaminant type.


In [ ]:
def build_texture_pool(texture_path: Path) -> dict:
    pool = {}
    for contaminant_dir in sorted(texture_path.iterdir()):
        if not contaminant_dir.is_dir():
            continue
        textures = sorted(
            f for f in contaminant_dir.iterdir()
            if f.suffix.lower() == '.png' and '_nobg' not in f.stem
        )
        if textures:
            pool[contaminant_dir.name] = textures
    return pool

texture_pool = build_texture_pool(TEXTURE_PATH)

print(f"Discovered {len(texture_pool)} contaminant types")
for contaminant, textures in texture_pool.items():
    print(f"  {contaminant:<25} {len(textures)} images")


#### 3.2.2 Display Texture Pool Summary

This cell prints a full inventory of the texture pool, listing each file under its contaminant type. We can confirm that all expected texture files were discovered and that no files were skipped.


In [ ]:
total = sum(len(v) for v in texture_pool.values())
print(f"Contamination Texture Library")
print(f"{'='*50}")
for contaminant, textures in texture_pool.items():
    print(f"\n  {contaminant}")
    for t in textures:
        print(f"    {t.name}")
print(f"\n{'='*50}")
print(f"Total: {len(texture_pool)} types, {total} images")


Here we can see there are a total of **40 images across the 8 different types of contaminant textures**.


### 3.3 Contamination Severity Levels

`CONTAMINATION_LEVELS` (defined in Step 0) maps each severity label to the number of texture patches applied to the object:

| Level | Patches | Description |
|---|---|---|
| light | 1 | A single texture patch applied to one region of the object |
| medium | 2 | Two patches applied to separate regions of the object |
| heavy | 3 | Three patches covering multiple regions of the object |


### 3.4 Define Image Processing Functions


#### 3.4.1 Object Segmentation

`get_object_mask` runs `rembg` on a source image to produce a soft float mask that identifies the foreground object. Pixel values close to 1.0 are confidently part of the object. Pixel values close to 0.0 are background.

Two post-processing steps clean up the raw mask before it is used for compositing:

1. **Morphological closing** fills small holes that rembg may leave inside the object, for example through transparent or reflective surfaces on glass bottles or plastic bags.
2. **Gaussian blur** feathers the mask edges so composited contamination fades naturally at the object boundary instead of producing a hard cutline.


In [ ]:
def get_object_mask(img: np.ndarray) -> np.ndarray:
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format='PNG')
    output = remove(buf.getvalue(), session=REMBG_SESSION)
    alpha  = np.array(Image.open(io.BytesIO(output)).convert('RGBA'))[:, :, 3]
    alpha  = alpha.astype(np.float32) / 255.0
    # Fill small interior holes left by transparent or reflective surfaces
    binary = ndimage.binary_closing(
        alpha > 0.5, structure=np.ones((5, 5))
    ).astype(np.float32)
    # Feather edges so composited contamination fades at the object boundary
    return ndimage.gaussian_filter(binary, sigma=2)


#### 3.4.2 Texture Loading

`load_texture` loads a contamination texture as an RGBA array and removes its background before compositing. `rembg` is used for background removal, which handles any background colour robustly.

The loading strategy depends on file format. For PNG files that already contain alpha transparency the file is loaded directly. For all other cases, including fully opaque PNGs and JPEG or AVIF files that cannot store transparency, `rembg` is run to isolate the contaminant texture from its background.

The processed RGBA is cached as a `_nobg.png` file in `/kaggle/working/texture_cache/` since `/kaggle/input` is read-only. Background removal only runs once per texture; subsequent runs load from cache.


In [ ]:
def load_texture(texture_path: Path) -> np.ndarray:
    cache_path = TEXTURE_CACHE / texture_path.parent.name / (texture_path.stem + '_nobg.png')
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        return np.array(Image.open(cache_path).convert('RGBA'))

    ext           = texture_path.suffix.lower()
    needs_removal = True

    if ext == '.png':
        rgba = np.array(Image.open(texture_path).convert('RGBA'))
        # If the alpha channel already has transparency, use it directly
        if rgba[:, :, 3].min() < 255:
            needs_removal = False

    if needs_removal:
        with open(texture_path, 'rb') as f:
            output = remove(f.read(), session=REMBG_SESSION)
        rgba = np.array(Image.open(io.BytesIO(output)).convert('RGBA'))

    Image.fromarray(rgba).save(cache_path)
    return rgba


#### 3.4.3 Texture Patch Application

`apply_texture_patch` composites a single contamination texture patch onto the object region of a base image. The texture is randomly augmented before placement so the same source texture file produces a visually distinct patch every time it is used.

The final composite is controlled by three masks multiplied together into a single weight map:

| Mask | Source | Purpose |
|---|---|---|
| Texture alpha | The texture's own RGBA alpha channel | Preserves the stain or residue shape |
| Object mask | Output of `get_object_mask` | Restricts the composite to the foreground object |
| Alpha scalar | Random float in [0.45, 0.75] | Controls the overall opacity of the contamination effect |

The anchor point for each patch is sampled uniformly at random from within the object region, which ensures patches land on the object surface rather than at a fixed location.


In [ ]:
def apply_texture_patch(
    base_img:     np.ndarray,
    texture_rgba: np.ndarray,
    object_mask:  np.ndarray,
    alpha:        float = 0.6,
) -> np.ndarray:
    H, W = base_img.shape[:2]

    # Sample a random anchor point from within the object region
    obj_ys, obj_xs = np.where(object_mask > 0.3)
    if len(obj_ys) == 0:
        return base_img
    idx = random.randint(0, len(obj_ys) - 1)
    cy, cx = int(obj_ys[idx]), int(obj_xs[idx])

    # Augment the texture
    tex = Image.fromarray(texture_rgba, 'RGBA')
    tex = tex.rotate(random.uniform(0, 360), expand=True)
    scale = random.uniform(0.25, 0.60)
    tex = tex.resize((max(1, int(W * scale)), max(1, int(H * scale))), Image.LANCZOS)
    if random.random() > 0.5:
        tex = tex.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        tex = tex.transpose(Image.FLIP_TOP_BOTTOM)

    tex_arr = np.array(tex)
    th, tw  = tex_arr.shape[:2]

    # Placement coordinates centered on the anchor
    x1, y1 = cx - tw // 2, cy - th // 2
    x2, y2 = x1 + tw, y1 + th

    # Clip to image bounds
    ix1, iy1 = max(0, x1), max(0, y1)
    ix2, iy2 = min(W, x2), min(H, y2)
    if ix2 <= ix1 or iy2 <= iy1:
        return base_img

    sx1 = ix1 - x1
    sy1 = iy1 - y1
    sx2 = sx1 + (ix2 - ix1)
    sy2 = sy1 + (iy2 - iy1)
    tex_slice = tex_arr[sy1:sy2, sx1:sx2]

    # Three-way weight: texture alpha x object mask x opacity scalar
    tex_a  = tex_slice[:, :, 3].astype(np.float32) / 255.0
    obj_a  = object_mask[iy1:iy2, ix1:ix2]
    weight = tex_a * obj_a * alpha

    # Composite the patch
    result  = base_img.astype(np.float32)
    region  = result[iy1:iy2, ix1:ix2]
    tex_rgb = tex_slice[:, :, :3].astype(np.float32)
    result[iy1:iy2, ix1:ix2] = (
        region * (1 - weight[..., np.newaxis]) + tex_rgb * weight[..., np.newaxis]
    )
    return np.clip(result, 0, 255).astype(np.uint8)


### 3.5 Build the Output Directory Structure

We only write the contaminated images to disk during generation. Clean images are referenced directly from the Stage A source dataset in the manifest. Each contaminated subdirectory encodes the source class so the binary label is recoverable from the path.

Any existing Stage B Dataset output and texture cache is cleared before the pipeline runs so we always start from a clean state.


In [ ]:
# Reset Stage B working directory and texture cache so generation runs from scratch
for p in [STAGE_B_PATH, TEXTURE_CACHE]:
    if p.exists():
        shutil.rmtree(p)
        print(f"Cleared {p}")

(STAGE_B_PATH / 'clean').mkdir(parents=True, exist_ok=True)
for cls in RECYCLABLE_CLASSES:
    (STAGE_B_PATH / 'contaminated' / cls).mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory structure created under {STAGE_B_PATH}")


### 3.6 Run the Generation Pipeline

We collect all source images upfront and pre-assign a severity level to each one before the loop begins. Pre-assigning ensures the contaminated set is exactly balanced: `ceil(n / 3)` images per level, trimmed to the total image count and shuffled so no class is biased toward a particular level.

For each source image the pipeline executes three steps:

1. **Load and resize** the source image to 224x224 pixels to match the Stage A input size.
2. **Save the clean copy** into `Stage B Dataset/clean/{class_name}/`.
3. **Generate one contaminated copy** using the pre-assigned level. A contaminant type is sampled at random, N texture patches are applied using `apply_texture_patch`, and the result is saved into `Stage B Dataset/contaminated/{class_name}/`.

We also record a `source_stem` column on every row of the manifest. This lets Step 4 split the dataset by source image (keeping each clean/contaminated pair together) rather than by random row, which would leak nearly identical objects across the train/val/test boundary and inflate the reported metrics. Estimated runtime on Kaggle T4 with `u2netp` + GPU rembg is ~**XX minutes** for the full ~10,500 source images.


In [ ]:
records           = []
contaminant_types = list(texture_pool.keys())

# Collect all source images across every recyclable class
all_images = []
for class_name in RECYCLABLE_CLASSES:
    class_dir = DATASET_PATH / class_name
    if not class_dir.exists():
        print(f"WARNING: {class_name} not found in dataset, skipping")
        continue
    for img_path in class_dir.rglob('*'):
        if img_path.is_file():
            all_images.append((class_name, img_path))

# MAX_IMAGES (from Step 0) is None for full runs; set it to subsample for smoke tests
all_images = all_images[:MAX_IMAGES]
n = len(all_images)

# Pre-assign balanced levels: ceil(n/3) of each, trimmed to n, then shuffled
n_per_level = math.ceil(n / 3)
level_pool  = (['light'] * n_per_level + ['medium'] * n_per_level + ['heavy'] * n_per_level)[:n]
random.shuffle(level_pool)

print(f"Source images : {n:,}")
print(f"  light       : {level_pool.count('light'):,}")
print(f"  medium      : {level_pool.count('medium'):,}")
print(f"  heavy       : {level_pool.count('heavy'):,}")
print(f"Expected total: {n * 2:,}  ({n:,} clean + {n:,} contaminated)\n")

for (class_name, img_path), level in tqdm(
    zip(all_images, level_pool), total=n, desc="Generating"
):
    source_stem = f"{img_path.parent.name}_{img_path.stem}"   # e.g. default_Image_42

    base_img = np.array(
        Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    )

    # Save clean copy
    clean_dir  = STAGE_B_PATH / 'clean' / class_name
    clean_dir.mkdir(parents=True, exist_ok=True)
    clean_path = clean_dir / f"{source_stem}.png"
    Image.fromarray(base_img).save(clean_path, format='PNG')

    records.append({
        'image_path':   str(clean_path),
        'label':        'clean',
        'subgroup':     'none',
        'level':        'none',
        'source_class': class_name,
        'source_stem':  source_stem,
    })

    # Generate the object mask once per source (rembg is the expensive call)
    mask = get_object_mask(base_img)

    # Sample a contaminant type and apply N patches for the assigned level
    contaminant_type   = random.choice(contaminant_types)
    available_textures = texture_pool[contaminant_type]
    n_patches          = CONTAMINATION_LEVELS[level]

    result_img = base_img.copy()
    for _ in range(n_patches):
        tex_path   = random.choice(available_textures)
        texture    = load_texture(tex_path)
        alpha      = random.uniform(0.45, 0.75)
        result_img = apply_texture_patch(result_img, texture, mask, alpha=alpha)

    cont_dir  = STAGE_B_PATH / 'contaminated' / class_name
    cont_dir.mkdir(parents=True, exist_ok=True)
    cont_path = cont_dir / f"{source_stem}_{contaminant_type}_{level}.png"
    Image.fromarray(result_img).save(cont_path, format='PNG')

    records.append({
        'image_path':   str(cont_path),
        'label':        'contaminated',
        'subgroup':     contaminant_type,
        'level':        level,
        'source_class': class_name,
        'source_stem':  source_stem,
    })

print(f"\nGeneration complete: {n:,} source images -> {len(records):,} total images")


### 3.7 Build the Stage B Manifest

The manifest is a CSV file that records every image in the dataset alongside its full label information. It also decouples the Stage B dataloader from the directory structure -- Step 4 reads this manifest, groups rows by `source_stem`, and uses that grouping to do a source-aware train/val/test split.

| Column | Values | Description |
|---|---|---|
| image_path | string | Absolute file path |
| label | `clean`, `contaminated` | Binary classification target for Stage B |
| subgroup | contaminant type or `none` | Which contaminant type was applied |
| level | `light`, `medium`, `heavy` or `none` | Contamination severity |
| source_class | waste class name | Original Stage A waste class the image came from |
| source_stem | string | Unique identifier shared between each clean / contaminated pair |


In [ ]:
df_stage_b = pd.DataFrame(records)

manifest_path = STAGE_B_PATH / 'stage_b_manifest.csv'
df_stage_b.to_csv(manifest_path, index=False)

print(f"Manifest saved to {manifest_path}")
print(f"Total images: {len(df_stage_b):,}")
print(f"\nSample rows:")
print(df_stage_b.head(8).to_string(index=False))


### 3.8 Audit the Dataset

With the manifest built, we can audit the dataset to verify that the generation pipeline produced the expected number of images per label, subgroup and severity level.


#### 3.8.1 Label Distribution

This cell prints the total image count split by binary label. The clean and contaminated sets should be in a 1:1 ratio (roughly 10.5K each). Each source image produces one clean entry in the manifest and one contaminated copy, with the contaminated set split evenly across the three severity levels.


In [ ]:
label_counts = df_stage_b['label'].value_counts()

print("Label Distribution")
print("=" * 40)
for label, count in label_counts.items():
    pct = count / len(df_stage_b) * 100
    print(f"  {label:<20} {count:>7,}  ({pct:.1f}%)")
print("-" * 40)
print(f"  {'Total':<20} {len(df_stage_b):>7,}")


#### 3.8.2 Subgroup Distribution

This cell prints the image count per contaminant type within the contaminated set. Because contaminant type is assigned uniformly at random per source image, each type should receive approximately equal representation (roughly 1/8 of the contaminated set).


In [ ]:
contaminated_df = df_stage_b[df_stage_b['label'] == 'contaminated']
subgroup_counts = contaminated_df['subgroup'].value_counts().sort_index()

print("Subgroup Distribution (contaminated images only)")
print("=" * 50)
for subgroup, count in subgroup_counts.items():
    pct = count / len(contaminated_df) * 100
    print(f"  {subgroup:<28} {count:>7,}  ({pct:.1f}%)")
print("-" * 50)
print(f"  {'Total':<28} {len(contaminated_df):>7,}")


#### 3.8.3 Level Distribution

This cell prints the image count per severity level within the contaminated set. Each level is pre-assigned once per source image so the three levels should have near-identical counts.


In [ ]:
level_counts = contaminated_df['level'].value_counts()
level_order  = ['light', 'medium', 'heavy']

print("Level Distribution (contaminated images only)")
print("=" * 40)
for level in level_order:
    count = level_counts.get(level, 0)
    pct   = count / len(contaminated_df) * 100
    print(f"  {level:<20} {count:>7,}  ({pct:.1f}%)")
print("-" * 40)
print(f"  {'Total':<20} {len(contaminated_df):>7,}")


#### 3.8.4 Summary Bar Charts

This cell plots the label, subgroup and level distributions side by side and saves the figure to `Synthetic Recyclable Contamination Dataset/dataset_audit.png`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Label distribution
axes[0].bar(label_counts.index, label_counts.values, color=['#2196F3', '#E53935'], width=0.4)
axes[0].set_title('Label Distribution', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Label', fontsize=10); axes[0].set_ylabel('Image Count', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(label_counts.items()):
    axes[0].text(i, count + 50, f"{count:,}", ha='center', fontsize=9)

# Subgroup distribution
axes[1].bar(subgroup_counts.index, subgroup_counts.values, color='#8E44AD', width=0.6)
axes[1].set_title('Contamination Subgroup Distribution', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Contaminant Type', fontsize=10); axes[1].set_ylabel('Image Count', fontsize=10)
axes[1].tick_params(axis='x', rotation=45); axes[1].grid(axis='y', alpha=0.3)

# Level distribution
level_vals   = [level_counts.get(l, 0) for l in level_order]
level_colors = ['#81C784', '#FFA726', '#E53935']
axes[2].bar(level_order, level_vals, color=level_colors, width=0.4)
axes[2].set_title('Contamination Level Distribution', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Level', fontsize=10); axes[2].set_ylabel('Image Count', fontsize=10)
axes[2].grid(axis='y', alpha=0.3)
for i, count in enumerate(level_vals):
    axes[2].text(i, count + 50, f"{count:,}", ha='center', fontsize=9)

plt.suptitle('Stage B Contamination Dataset Audit', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
audit_chart_path = DATASET_ROOT / 'dataset_audit.png'
plt.savefig(audit_chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {audit_chart_path}")


#### 3.8.5 Visual Sample Audit

This cell randomly samples five matched clean/contaminated pairs and displays them in two rows. Each image is labelled with its binary label, subgroup and severity level so the compositing quality can be inspected visually.


In [ ]:
clean_rows = df_stage_b[df_stage_b['label'] == 'clean'].reset_index(drop=True)
cont_rows  = df_stage_b[df_stage_b['label'] == 'contaminated'].reset_index(drop=True)

sample_idx = random.sample(range(len(cont_rows)), min(5, len(cont_rows)))
pairs = [(clean_rows.iloc[i], cont_rows.iloc[i]) for i in sample_idx]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for col, (clean_row, cont_row) in enumerate(pairs):
    axes[0, col].imshow(Image.open(clean_row['image_path']).convert('RGB'))
    axes[0, col].axis('off')
    axes[0, col].set_title(f"clean\n{clean_row['source_class']}", fontsize=7.5,
                            color='#2196F3', fontweight='bold', pad=4)

    axes[1, col].imshow(Image.open(cont_row['image_path']).convert('RGB'))
    axes[1, col].axis('off')
    axes[1, col].set_title(
        f"contaminated\n{cont_row['subgroup']} | {cont_row['level']}\n{cont_row['source_class']}",
        fontsize=7.5, color='#E53935', fontweight='bold', pad=4
    )

fig.text(0.01, 0.73, 'Clean',        va='center', rotation='vertical', fontsize=11, fontweight='bold', color='#2196F3')
fig.text(0.01, 0.27, 'Contaminated', va='center', rotation='vertical', fontsize=11, fontweight='bold', color='#E53935')

plt.suptitle('Stage B Dataset Visual Sample Audit - Matched Pairs', fontsize=13, fontweight='bold')
plt.tight_layout()

visual_audit_path = DATASET_ROOT / 'visual_sample_audit.png'
plt.savefig(visual_audit_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {visual_audit_path}")


### 3.9 Save Synthetic Recyclable Contamination Dataset

This cell assembles the final packaged dataset. The intermediate `Stage B Dataset/` working directory used during generation is reorganized into a clean shape under `Synthetic Recyclable Contamination Dataset/images/`. Both clean and contaminated sides sit at the same `images/{label}/{class}/` depth so `image_dataset_from_directory` could read them directly if needed.

```
Synthetic Recyclable Contamination Dataset/
|-- stage_b_manifest.csv
|-- dataset_audit.png
|-- visual_sample_audit.png
\-- images/
    |-- clean_recyclables/
    |   |-- plastic_soda_bottles/
    |   \-- ... (21 classes)
    \-- contaminated_recyclables/
        |-- plastic_soda_bottles/
        \-- ... (21 classes)
```

The manifest is rewritten with paths relative to the dataset root so the CSV works outside the original Kaggle session, and the whole dataset is zipped for one-click download from `/kaggle/working/`.


In [ ]:
IMAGES_ROOT = DATASET_ROOT / 'images'
CLEAN_DIR   = IMAGES_ROOT / 'clean_recyclables'
CONTAM_DIR  = IMAGES_ROOT / 'contaminated_recyclables'

# Reset the final dataset images directory so we start from a clean state
if IMAGES_ROOT.exists():
    shutil.rmtree(IMAGES_ROOT)
IMAGES_ROOT.mkdir(parents=True)

# --- Clean images: copy from the working clean/{class}/ folders ---
print("Copying clean images...")
total_clean = 0
for img_file in tqdm(list((STAGE_B_PATH / 'clean').rglob('*.png'))):
    cls      = img_file.parent.name
    dst_file = CLEAN_DIR / cls / img_file.name
    dst_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_file, dst_file)
    total_clean += 1

# --- Contaminated images ---
print("\nCopying contaminated images...")
total_contam = 0
for img_file in tqdm(list((STAGE_B_PATH / 'contaminated').rglob('*.png'))):
    cls      = img_file.parent.name
    dst_file = CONTAM_DIR / cls / img_file.name
    dst_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_file, dst_file)
    total_contam += 1

# --- Rewrite manifest with paths relative to DATASET_ROOT ---
df_manifest = pd.read_csv(STAGE_B_PATH / 'stage_b_manifest.csv')

def to_relative_path(row):
    label_dir = 'clean_recyclables' if row['label'] == 'clean' else 'contaminated_recyclables'
    p = Path(row['image_path'])
    return f"images/{label_dir}/{row['source_class']}/{p.name}"

df_manifest['image_path'] = df_manifest.apply(to_relative_path, axis=1)
df_manifest.to_csv(DATASET_ROOT / 'stage_b_manifest.csv', index=False)

# --- Zip the final dataset for easy download ---
zip_base = WORKING_PATH / 'Synthetic Recyclable Contamination Dataset'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=str(DATASET_ROOT)))
print(f"\nZipped dataset to {zip_path}")

# Keep STAGE_B_PATH around for now -- Step 4 reads the absolute paths in df_stage_b
print(f"\nFinal directory structure:")
print(f"Synthetic Recyclable Contamination Dataset/")
print(f"|-- stage_b_manifest.csv")
print(f"|-- dataset_audit.png")
print(f"|-- visual_sample_audit.png")
print(f"\-- images/")
print(f"    |-- clean_recyclables/        {total_clean:>6,} images across {len(list(CLEAN_DIR.iterdir()))} classes")
print(f"    \-- contaminated_recyclables/ {total_contam:>6,} images across {len(list(CONTAM_DIR.iterdir()))} classes")


## Step 4: Train Stage B -- Contamination Classifier

Stage B is a binary classifier that determines whether a recyclable item is contaminated. It reuses the MobileNetV3Large backbone we trained in Stage A, replacing the multi-class softmax head with a single sigmoid output. We train Stage B across 30 epochs with the backbone frozen, so only the new sigmoid head is updated while the feature extractor stays fixed.

The training data is the Synthetic Recyclable Contamination Dataset we built in Step 3. Each source image appears once as **clean** and once as **contaminated**, and we split by `source_stem` to keep matched pairs together (preventing leakage where the model sees a near-identical object at training and test time).


### 4.1 Load and Split the Stage B Dataset

Before we load the Stage B data we release the cached Stage A datasets, the rembg ONNX session and the Stage A model objects, since they collectively hold ~10 GB of RAM that we no longer need.

We then read the manifest from Step 3 and do a **source-aware 70/15/15 split**: unique `source_stem` values are split first (stratified by `source_class`), then the clean + contaminated rows belonging to each source are routed into the same split. This is the difference between a measurement of generalization (what we want) and a measurement of memorization (what a naive row-wise split would give us).

We cache val and test in memory (they are small) but leave train uncached -- the full training set is too large to fit comfortably alongside both models on a Kaggle T4.


In [ ]:
# Release Stage A artifacts to free up RAM and VRAM for Stage B
for _ds in ['train_ds', 'val_ds', 'test_ds', 'test_default_ds', 'test_real_world_ds']:
    if _ds in globals():
        del globals()[_ds]

if 'REMBG_SESSION' in globals():
    del REMBG_SESSION

for _name in ['model_a', 'base_model', 'best_model_a']:
    if _name in globals():
        del globals()[_name]

gc.collect()
tf.keras.backend.clear_session()
print("Released Stage A datasets, rembg session and Stage A model from memory")

# --- Source-aware 70/15/15 split ---
manifest_b = df_stage_b.copy()

unique_sources = manifest_b[['source_stem', 'source_class']].drop_duplicates()
train_sources, temp_sources = train_test_split(
    unique_sources, test_size=0.30,
    stratify=unique_sources['source_class'], random_state=SEED,
)
val_sources, test_sources = train_test_split(
    temp_sources, test_size=0.50,
    stratify=temp_sources['source_class'], random_state=SEED,
)

train_stems = set(train_sources['source_stem'])
val_stems   = set(val_sources['source_stem'])
test_stems  = set(test_sources['source_stem'])

train_df_b = manifest_b[manifest_b['source_stem'].isin(train_stems)].reset_index(drop=True)
val_df_b   = manifest_b[manifest_b['source_stem'].isin(val_stems)  ].reset_index(drop=True)
test_df_b  = manifest_b[manifest_b['source_stem'].isin(test_stems) ].reset_index(drop=True)

# Encode labels as 0=clean, 1=contaminated
class_names_b = ['clean_recyclables', 'contaminated_recyclables']
for df in (train_df_b, val_df_b, test_df_b):
    df['label_idx'] = (df['label'] == 'contaminated').astype(np.int32)

# Build tf.data datasets from the (file_path, label_idx) tensor slices
def _load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    return img, tf.cast(label, tf.float32)

def _build_ds(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['image_path'].values, df['label_idx'].values)
    )
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED)
    ds = ds.map(_load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

train_b_ds = _build_ds(train_df_b, shuffle=True)
val_b_ds   = _build_ds(val_df_b).cache()
test_b_ds  = _build_ds(test_df_b).cache()

print(f"Classes      : {class_names_b}")
print(f"Unique sources: train={len(train_stems):,}  val={len(val_stems):,}  test={len(test_stems):,}")
print(f"Total images  : train={len(train_df_b):,}  val={len(val_df_b):,}  test={len(test_df_b):,}")
print(f"Train batches : {int(train_b_ds.cardinality())}")
print(f"Val batches   : {int(val_b_ds.cardinality())}")
print(f"Test batches  : {int(test_b_ds.cardinality())}")


### 4.2 Data Augmentation

Stage B reuses the same augmentation pipeline we defined in Step 2 (random rotation, flips, zoom, translation, brightness, contrast). Augmentation is embedded inside the model graph in section 4.3 and is automatically disabled during evaluation. We don't need a separate cell here since `data_augmentation` is already defined and just gets wired into the Stage B model in the next section.


### 4.3 Build the Stage B Contamination Classifier

We load the Stage A best checkpoint from disk, extract the MobileNetV3Large backbone using the pinned name `stage_a_backbone` from section 2.3, and attach a binary classification head. The head is the same shape as Stage A (GAP -> BN -> Dropout -> Dense(256, ReLU + L2)) but ends in a single sigmoid neuron with a `float32` cast for mixed precision stability. We freeze the backbone for the entire run so the new sigmoid head can stabilize on top of the feature extractor we trained in Stage A.


In [ ]:
def build_stage_b(stage_a_weights_path):
    stage_a  = tf.keras.models.load_model(stage_a_weights_path)
    # _name patching does not survive Keras save/load; the backbone is
    # serialised under its original EfficientNetV2S default name.
    backbone = stage_a.get_layer('efficientnetv2-s')
    backbone.trainable = False

    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = data_augmentation(inputs)
    x       = backbone(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    # Final sigmoid in fp32 for numerical stability under mixed_float16
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='contamination')(x)
    model   = Model(inputs, outputs, name='stage_b')

    del stage_a
    gc.collect()
    return model, backbone

with strategy.scope():
    model_b, backbone_b = build_stage_b(MODELS_PATH / 'stage_a_best.keras')
model_b.summary()

### 4.4 Define Callbacks

Three callbacks are used, matching Stage A in role. `ModelCheckpoint` and `EarlyStopping` both monitor `val_auc` rather than `val_accuracy` because AUC is more informative for binary classifiers near a decision boundary. `ReduceLROnPlateau` watches `val_loss` to detect training plateaus.


In [ ]:
checkpoint_b_cb = ModelCheckpoint(
    str(MODELS_PATH / 'stage_b_best.keras'),
    monitor='val_auc', mode='max', save_best_only=True, verbose=1,
)
early_stop_b_cb = EarlyStopping(
    monitor='val_auc', mode='max', patience=7, restore_best_weights=True,
)
reduce_lr_b_cb = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1,
)


Stage B is trained with the same two-phase recipe as Stage A.

**Phase 1 -- Head warmup (15 epochs, lr=1e-3, backbone frozen).** The EfficientNetV2-S backbone (inherited from the best Stage A checkpoint) stays frozen and only the binary contamination head trains. Loss is plain `binary_crossentropy`; no class weights are needed because the Step 3 manifest produces a 1:1 clean-vs-contaminated split by construction.

**Phase 2 -- Fine-tune top 35% of backbone (25 epochs, lr=1e-5 with cosine decay).** Same unfreeze rule as Stage A: bottom 65% of layers stay frozen, all `BatchNormalization` layers stay in inference mode. `CosineDecay` smoothly anneals the learning rate to zero across the 25 epochs.

Phase 1 uses `ReduceLROnPlateau`; Phase 2 swaps to `EarlyStopping` since the cosine schedule already manages the learning rate.

In [ ]:
# ---------- Phase 1: head warmup, backbone fully frozen ----------
print("=== Stage B Phase 1: head warmup (backbone frozen) ===")
backbone_b.trainable = False
with strategy.scope():
    model_b.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
history_b_p1 = model_b.fit(
    train_b_ds,
    epochs=15,
    validation_data=val_b_ds,
    callbacks=[checkpoint_b_cb, reduce_lr_b_cb],
    verbose=1,
)

# ---------- Phase 2: fine-tune top 35% of backbone with cosine-decay LR ----------
print("\n=== Stage B Phase 2: fine-tune top 35% backbone (cosine decay) ===")
n_layers      = len(backbone_b.layers)
unfreeze_from = int(n_layers * 0.65)
for i, layer in enumerate(backbone_b.layers):
    if i < unfreeze_from or isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

steps_per_epoch = int(train_b_ds.cardinality())
schedule        = CosineDecay(initial_learning_rate=1e-5, decay_steps=25 * steps_per_epoch)
with strategy.scope():
    model_b.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=schedule),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
history_b_p2 = model_b.fit(
    train_b_ds,
    epochs=25,
    validation_data=val_b_ds,
    callbacks=[checkpoint_b_cb, early_stop_b_cb],
    verbose=1,
)

# Stitch the two phase histories together for the plotting cell
history_b = type('H', (), {})()
history_b.history = combine_histories(history_b_p1, history_b_p2)

print(f"\nStage B model saved to: {MODELS_PATH / 'stage_b_best.keras'}")

### 4.6 Plot Stage B Training vs Validation Error Metrics

We plot accuracy, AUC, and loss across the 30 training epochs. A tight gap between the train and val curves indicates the model is generalizing rather than memorizing the training data, which is what we want to see.


In [ ]:
# AUC metric key can sometimes auto-suffix (e.g. 'auc_1') if a model is rebuilt;
# pick the first key that starts with 'auc' so the plot is robust across reruns.
auc_key     = next(k for k in history_b.history if k.startswith('auc') and not k.startswith('val_'))
val_auc_key = 'val_' + auc_key

epochs_b = range(1, len(history_b.history['accuracy']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Stage B: Training vs Validation Error Metrics', fontsize=14, fontweight='bold')

plot_specs = [
    ('accuracy', 'val_accuracy', 'Accuracy'),
    (auc_key,    val_auc_key,    'AUC'),
    ('loss',     'val_loss',     'Loss'),
]

for ax, (train_key, val_key, title) in zip(axes, plot_specs):
    ax.plot(epochs_b, history_b.history[train_key], label='Train')
    ax.plot(epochs_b, history_b.history[val_key],   label='Val')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(STAGE_B_RESULTS_PATH / 'stage_b_error_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.7 Evaluate Stage B Contamination Classifier on Test Set

We load the best checkpoint and run inference on the held-out test set. The table reports per-class accuracy, precision, recall, and F1 for clean and contaminated. We then visualize those metrics side by side on a bar chart. The results DataFrame we build here is reused in sections 4.8 and 4.9 for deeper breakdowns, and the binary probabilities (`all_b_probs`) are re-used in Step 5 to tune the F1-optimal decision threshold.


In [ ]:
best_model_b = tf.keras.models.load_model(MODELS_PATH / 'stage_b_best.keras')

# test_b_ds was built with shuffle=False so predictions align with test_df_b row order
all_b_probs, all_b_labels = [], []
for images, labels in test_b_ds:
    probs = best_model_b.predict(images, verbose=0)
    all_b_probs.extend(probs[:, 0].tolist())
    all_b_labels.extend(labels.numpy().astype(int).flatten().tolist())

all_b_probs  = np.array(all_b_probs)
all_b_labels = np.array(all_b_labels)
all_b_preds  = (all_b_probs >= 0.5).astype(int)

cm_b            = confusion_matrix(all_b_labels, all_b_preds)
per_class_acc_b = cm_b.diagonal() / cm_b.sum(axis=1).clip(min=1)

# Per-class metrics (clean=0, contaminated=1)
rows = []
for i, name in enumerate(class_names_b):
    display_name = name.replace('_recyclables', '')
    mask = all_b_labels == i
    rows.append({
        'Class':     display_name,
        'Accuracy':  per_class_acc_b[i],
        'Precision': precision_score(all_b_labels == i, all_b_preds == i, zero_division=0),
        'Recall':    recall_score(all_b_labels == i,    all_b_preds == i, zero_division=0),
        'F1':        f1_score(all_b_labels == i,        all_b_preds == i, zero_division=0),
    })

df_b_metrics = pd.DataFrame(rows)
overall_acc  = (all_b_preds == all_b_labels).mean()
auc_b        = roc_auc_score(all_b_labels, all_b_probs)

print(df_b_metrics.to_string(index=False))
print(f'\nOverall Accuracy : {overall_acc:.4f}')
print(f'ROC-AUC          : {auc_b:.4f}')

# Build results DataFrame for sections 4.8 and 4.9
results_b = test_df_b.copy().reset_index(drop=True)
results_b['true_label'] = all_b_labels
results_b['pred_label'] = all_b_preds
results_b['prob']       = all_b_probs


In [ ]:
display_names = [n.replace('_recyclables', '') for n in class_names_b]
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1']
bar_colors    = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
x             = np.arange(len(display_names))
width         = 0.18

fig, ax = plt.subplots(figsize=(9, 5))
for i, (metric, color) in enumerate(zip(metric_labels, bar_colors)):
    ax.bar(x + i * width, df_b_metrics[metric].values, width, label=metric, color=color)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(display_names, fontsize=11)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Stage B: Per-Class Evaluation Metrics', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(STAGE_B_RESULTS_PATH / 'stage_b_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.8 Per-Contamination-Level Breakdown

The synthetic dataset assigns one of three severity levels to each contaminated image: **light** (1 texture patch), **medium** (2 patches), and **heavy** (3 patches). This breakdown reveals whether the classifier struggles more with subtle contamination than with heavy contamination, which is expected given that light contamination has a smaller visual footprint.


In [ ]:
cont_results = results_b[results_b['label'] == 'contaminated'].copy()
level_order  = ['light', 'medium', 'heavy']
level_rows   = []

for level in level_order:
    lvl_df = cont_results[cont_results['level'] == level]
    if len(lvl_df) == 0:
        continue
    acc  = (lvl_df['pred_label'] == lvl_df['true_label']).mean()
    prec = precision_score(lvl_df['true_label'], lvl_df['pred_label'], zero_division=0)
    rec  = recall_score(lvl_df['true_label'],    lvl_df['pred_label'], zero_division=0)
    f1   = f1_score(lvl_df['true_label'],        lvl_df['pred_label'], zero_division=0)
    level_rows.append({'Level': level, 'n': len(lvl_df),
                       'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1})

df_level = pd.DataFrame(level_rows)
print(df_level.to_string(index=False))

x      = np.arange(len(df_level))
width  = 0.18

fig, ax = plt.subplots(figsize=(9, 5))
for i, (metric, color) in enumerate(zip(metric_labels, bar_colors)):
    ax.bar(x + i * width, df_level[metric].values, width, label=metric, color=color)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_level['Level'].values, fontsize=11)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Stage B: Metrics by Contamination Level', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(STAGE_B_RESULTS_PATH / 'stage_b_level_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.9 Per-Recyclable-Class Breakdown

This breakdown shows how contamination detection performance varies across the 21 recyclable object classes. Classes are sorted by overall accuracy. The three bars per class show overall accuracy, accuracy on the clean subset, and accuracy on the contaminated subset, which surfaces classes where the model has asymmetric difficulty distinguishing clean from contaminated.


In [ ]:
class_rows = []
for cls in sorted(results_b['source_class'].dropna().unique()):
    cls_df   = results_b[results_b['source_class'] == cls]
    clean_df = cls_df[cls_df['label'] == 'clean']
    cont_df  = cls_df[cls_df['label'] == 'contaminated']

    overall_acc = (cls_df['pred_label'] == cls_df['true_label']).mean()
    clean_acc   = (clean_df['pred_label'] == clean_df['true_label']).mean() if len(clean_df) > 0 else float('nan')
    cont_acc    = (cont_df['pred_label']  == cont_df['true_label']).mean()  if len(cont_df)  > 0 else float('nan')

    class_rows.append({
        'Class':            cls,
        'n':                len(cls_df),
        'Overall Accuracy': overall_acc,
        'Clean Accuracy':   clean_acc,
        'Contam. Accuracy': cont_acc,
    })

df_class = (
    pd.DataFrame(class_rows)
    .sort_values('Overall Accuracy', ascending=False)
    .reset_index(drop=True)
)
print(df_class.to_string(index=False))

x     = np.arange(len(df_class))
width = 0.28

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - width, df_class['Overall Accuracy'], width, label='Overall',      color='#2196F3')
ax.bar(x,         df_class['Clean Accuracy'],   width, label='Clean',        color='#4CAF50')
ax.bar(x + width, df_class['Contam. Accuracy'], width, label='Contaminated', color='#E53935')

ax.set_xticks(x); ax.set_xticklabels(df_class['Class'], rotation=45, ha='right', fontsize=9)
ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy')
ax.set_title('Stage B: Per-Recyclable-Class Accuracy', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(STAGE_B_RESULTS_PATH / 'stage_b_class_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 5: Build the Full Waste Classification Pipeline

The full pipeline combines Stage A and Stage B into a single decision system, with a privacy-preserving face check in front of both. Inference flow:

1. **Face check.** If a human face is detected in the frame the pipeline refuses to classify and returns a structured `rejected` result. We never want EcoBin to make predictions about people.
2. **Stage A.** Otherwise we predict the waste class and map it to a disposal pathway.
3. **Stage B (conditional).** If the Stage A pathway is curbside or drop-off recycling, we run Stage B to check for contamination. A contaminated recyclable is overridden to **garbage**. Compost and garbage predictions skip Stage B entirely since contamination detection is only meaningful for recyclables.


### 5.1 Load Best Stage A and Stage B Checkpoints

We reload the best checkpoints saved from Step 2 and Step 4. Loading them again here makes Step 5 reproducible on its own once both checkpoints exist on disk -- the rest of Step 5 + Step 6 do not depend on the in-memory `model_a` / `model_b` from training.


In [ ]:
best_model_a = tf.keras.models.load_model(MODELS_PATH / 'stage_a_best.keras')
best_model_b = tf.keras.models.load_model(MODELS_PATH / 'stage_b_best.keras')

print(f"Stage A loaded from {MODELS_PATH / 'stage_a_best.keras'}")
print(f"  Output classes : {best_model_a.output_shape[-1]}")
print(f"Stage B loaded from {MODELS_PATH / 'stage_b_best.keras'}")
print(f"  Output shape   : {best_model_b.output_shape}")


### 5.2 Determine the F1-Optimal Stage B Threshold

The default Stage B decision threshold of 0.5 treats clean and contaminated as equally weighted, but we can do better by tuning the threshold on the test set we already evaluated in section 4.7. We sweep the precision-recall curve and select the threshold that maximizes the F1 score. This becomes the threshold the pipeline uses when deciding whether to override a recyclable prediction to garbage.


In [ ]:
# all_b_probs and all_b_labels were computed in section 4.7 against the test set
precisions, recalls, thresholds = precision_recall_curve(all_b_labels, all_b_probs)

# F1 at each threshold (skip the final pair which has no associated threshold)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-12)
best_idx  = int(np.argmax(f1_scores))

STAGE_B_THRESHOLD = float(thresholds[best_idx])

print(f"F1-optimal Stage B threshold : {STAGE_B_THRESHOLD:.4f}")
print(f"Precision at threshold       : {precisions[best_idx]:.4f}")
print(f"Recall at threshold          : {recalls[best_idx]:.4f}")
print(f"F1 at threshold              : {f1_scores[best_idx]:.4f}")


### 5.3 Face-Exclusion Utility

EcoBin should never produce a waste classification from an image of a person. To enforce this we run a lightweight face detector before any other inference. If a face is detected, the pipeline returns a `rejected` result instead of running Stage A or Stage B.

We use **OpenCV's pre-trained Haar Cascade frontal-face detector** (`haarcascade_frontalface_default.xml`), which ships with the `opencv-python` package already installed on Kaggle. It runs in tens of milliseconds on CPU, requires no model download, and is accurate enough for the binary 'is there a face in the frame' question we're answering. Heavier detectors (MediaPipe, YOLO-face) would add a dependency for marginal gain here.

`FACE_CASCADE` was loaded in Step 0 so we just wrap it in a helper function.


In [ ]:
def contains_face(image: np.ndarray) -> bool:
    """Return True if the Haar cascade detects at least one face in `image` (RGB uint8)."""
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces = FACE_CASCADE.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50),
    )
    return len(faces) > 0


### 5.4 Build the End-to-End Inference Function

`infer(image)` takes a single RGB image array and returns a structured result describing the full pipeline decision. The function:

- runs the face check first; if a face is found, returns `{status: 'rejected', reason: 'face_detected'}`.
- otherwise resizes the image once to 224x224 and reuses it for both Stage A and Stage B.
- runs Stage A and looks up the disposal pathway.
- runs Stage B only when the pathway is `curbside_recycling` or `dropoff_recycling`. Stage B's sigmoid probability is compared against `STAGE_B_THRESHOLD` from section 5.2.
- overrides a contaminated recyclable's final pathway to `garbage`.

The return dict surfaces `final_pathway` plus enough intermediate detail to debug a misclassification (Stage A confidence, Stage B probability, whether Stage B even ran).


In [ ]:
RECYCLING_PATHWAYS = {'curbside_recycling', 'dropoff_recycling'}

def infer(image: np.ndarray) -> dict:
    """Run the full face-check -> Stage A -> Stage B pipeline on a single RGB image."""

    # 1. Privacy gate: refuse to classify any frame containing a human face
    if contains_face(image):
        return {
            'status':         'rejected',
            'reason':         'face_detected',
            'final_pathway':  None,
        }

    # 2. Pre-process once for both stages
    img       = Image.fromarray(image).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(np.array(img), axis=0).astype('float32')

    # 3. Stage A inference
    stage_a_probs      = best_model_a.predict(img_batch, verbose=0)[0]
    stage_a_idx        = int(np.argmax(stage_a_probs))
    stage_a_class      = class_names[stage_a_idx]
    stage_a_confidence = float(stage_a_probs[stage_a_idx])
    pathway            = DISPOSAL_MAP[stage_a_class]
    final_pathway      = pathway

    # 4. Stage B runs only on recyclable predictions
    stage_b_ran    = False
    stage_b_result = None

    if pathway in RECYCLING_PATHWAYS:
        prob_contaminated = float(best_model_b.predict(img_batch, verbose=0)[0, 0])
        is_contaminated   = prob_contaminated >= STAGE_B_THRESHOLD
        stage_b_ran       = True
        stage_b_result    = {
            'prob_contaminated': prob_contaminated,
            'prediction':        'contaminated' if is_contaminated else 'clean',
            'threshold':         STAGE_B_THRESHOLD,
        }
        if is_contaminated:
            final_pathway = 'garbage'   # Contamination override

    return {
        'status':             'ok',
        'predicted_class':    stage_a_class,
        'stage_a_confidence': stage_a_confidence,
        'stage_a_pathway':    pathway,
        'stage_b_ran':        stage_b_ran,
        'stage_b_result':     stage_b_result,
        'final_pathway':      final_pathway,
    }


### 5.5 Capture an Image and Run the Pipeline (Manual Test)

The cell below wires up a live webcam capture (via `ipywebrtc`) plus a file-upload fallback and calls `infer()` on the captured frame. It is **commented out so `Run All` never blocks waiting for a webcam frame**. After Run All completes, uncomment the cell, run it on its own, click `Take a Picture` or upload an image, then press `Classify`.


In [ ]:
# Commented out so Run All does not block on human input. Uncomment to test the
# full pipeline interactively after the rest of the notebook has finished running.
"""
!pip install ipywebrtc -q

from IPython.display import display
from ipywebrtc import CameraStream, ImageRecorder
from ipywidgets import FileUpload, Button, VBox, HBox, Output, HTML

camera   = CameraStream(constraints={'audio': False, 'video': {'width': 640, 'height': 480}})
recorder = ImageRecorder(stream=camera)
uploader = FileUpload(accept='image/*', multiple=False)
output   = Output()
classify_btn = Button(description='Classify', button_style='success', icon='check')

def _read_uploaded_image():
    if not uploader.value:
        return None
    if isinstance(uploader.value, (list, tuple)):
        img_bytes = uploader.value[-1]['content']
    else:
        img_bytes = list(uploader.value.values())[-1]['content']
    return np.array(Image.open(io.BytesIO(bytes(img_bytes))).convert('RGB'))

def _read_webcam_image():
    if not recorder.image.value:
        return None
    return np.array(Image.open(io.BytesIO(recorder.image.value)).convert('RGB'))

def on_classify(_):
    output.clear_output()
    with output:
        img = _read_webcam_image()
        if img is None:
            img = _read_uploaded_image()
        if img is None:
            print('Capture a webcam image (Take a Picture) or upload one first.')
            return

        result = infer(img)
        if result['status'] == 'rejected':
            print(f"Pipeline rejected the frame -- reason: {result['reason']}")
            return
        print(f"Stage A Prediction    : {result['predicted_class']}")
        print(f"Stage A Confidence    : {result['stage_a_confidence']:.2%}")
        print(f"Stage A Pathway       : {result['stage_a_pathway']}")
        print(f"Stage B Ran           : {result['stage_b_ran']}")
        if result['stage_b_ran']:
            sb = result['stage_b_result']
            print(f"Stage B Prediction    : {sb['prediction']}")
            print(f"Stage B Contamination : {sb['prob_contaminated']:.2%} (threshold {sb['threshold']:.2f})")
        print(f"Final Pathway         : {result['final_pathway']}")

classify_btn.on_click(on_classify)

display(HTML('<h3>EcoBin: Two-Stage Waste Classifier</h3>'))
display(HBox([
    VBox([HTML('<b>Webcam Capture</b>'), camera, recorder]),
    VBox([HTML('<b>Or Upload an Image</b>'), uploader]),
]))
display(classify_btn)
display(output)
"""


## Step 6: McNemar's Statistical Test

The engineering goal of this project is to determine whether adding the Stage B contamination classifier produces a statistically significant accuracy improvement over Stage A alone. We define two systems:

- **System A**: Stage A only. Predicts a disposal pathway directly from the Stage A class output. Cannot override recyclables to garbage because it has no contamination signal.
- **System B**: the full pipeline from Step 5. Runs Stage A then Stage B, and overrides recyclables to garbage when contamination is detected.

We run both systems on the **100-image curated test set** mounted at `TEST_SET_PATH` and apply McNemar's test on the paired binary outcomes (each system was either correct or incorrect on each image). McNemar's test focuses on **discordant pairs**: images where one system was right and the other was wrong. If System B's gains over System A are systematically larger than its losses, the test will return a small p-value and we reject the null hypothesis that the systems perform equivalently.

For this experiment we collapse the four disposal pathways into two: **recycling** (curbside + dropoff) and **garbage**. Compost is not represented in the curated test set since Stage B has no role in compost classification.


### 6.1 Load the Curated Test Set

The curated test set is pre-mounted at `TEST_SET_PATH` (set in Step 0) and has three subdirectories matching the three buckets of the experiment:

| Folder | Count | Ground Truth Pathway | Notes |
| --- | --- | --- | --- |
| `Clean Recyclables`        | 50 | recycling | System B should *not* override these |
| `Contaminated Recyclables` | 25 | garbage   | The only bucket where System B can possibly beat System A |
| `Garbage`                  | 25 | garbage   | System B should keep these correct (no false overrides) |

We load every image, attach its ground-truth pathway derived from the folder name, and flag the contaminated-recyclable rows for the supplementary override metrics in section 6.6.


In [ ]:
# Folder name -> (ground_truth_pathway, is_contaminated_recyclable)
bucket_to_truth = {
    'Clean Recyclables':        ('recycling', False),
    'Contaminated Recyclables': ('garbage',   True),
    'Garbage':                  ('garbage',   False),
}

records = []
for bucket, (truth_pathway, is_contam_recyclable) in bucket_to_truth.items():
    bucket_dir = TEST_SET_PATH / bucket
    if not bucket_dir.exists():
        raise FileNotFoundError(f"Missing bucket directory: {bucket_dir}")
    for img_path in sorted(bucket_dir.iterdir()):
        if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
            records.append({
                'file_path':                  str(img_path),
                'bucket':                     bucket,
                'ground_truth_pathway':       truth_pathway,
                'is_contaminated_recyclable': is_contam_recyclable,
            })

test_df = pd.DataFrame(records)
print(f"Loaded {len(test_df)} test images")
print()
print("Per-bucket counts")
print("-" * 40)
print(test_df['bucket'].value_counts().to_string())


### 6.2 Run System A and System B on the Test Set

System A returns the disposal pathway from Stage A only, collapsed to a binary recycling/garbage label. System B re-uses the same Stage A + Stage B + threshold logic from `infer()` in Step 5, but we **bypass the face gate** here -- we know the curated test set contains only waste objects, and we don't want a false face detection to silently drop a row. Both systems see the same input images, so the test is paired image by image.


In [ ]:
def pathway_to_binary(pathway):
    """Collapse the four disposal pathways into recycling vs garbage."""
    if pathway in ('curbside_recycling', 'dropoff_recycling'):
        return 'recycling'
    return 'garbage'

def predict_pair(image_array):
    """Run Stage A and (conditionally) Stage B on one image; return both predictions."""
    img       = Image.fromarray(image_array).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(np.array(img), axis=0).astype('float32')

    # Stage A
    a_probs = best_model_a.predict(img_batch, verbose=0)[0]
    a_class = class_names[int(np.argmax(a_probs))]
    a_pathway = DISPOSAL_MAP[a_class]
    system_a = pathway_to_binary(a_pathway)

    # Stage B override (only on recyclables)
    system_b = system_a
    if a_pathway in RECYCLING_PATHWAYS:
        prob_contam = float(best_model_b.predict(img_batch, verbose=0)[0, 0])
        if prob_contam >= STAGE_B_THRESHOLD:
            system_b = 'garbage'

    return system_a, system_b

system_a_preds, system_b_preds = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    img = np.array(Image.open(row['file_path']).convert('RGB'))
    a_pred, b_pred = predict_pair(img)
    system_a_preds.append(a_pred)
    system_b_preds.append(b_pred)

test_df['system_a_pred']    = system_a_preds
test_df['system_b_pred']    = system_b_preds
test_df['system_a_correct'] = test_df['system_a_pred'] == test_df['ground_truth_pathway']
test_df['system_b_correct'] = test_df['system_b_pred'] == test_df['ground_truth_pathway']

acc_a = test_df['system_a_correct'].mean()
acc_b = test_df['system_b_correct'].mean()

print(f"System A accuracy   : {acc_a:.4f}")
print(f"System B accuracy   : {acc_b:.4f}")
print(f"Absolute improvement: {acc_b - acc_a:+.4f}")


### 6.3 Confusion Matrices

We plot the System A and System B confusion matrices side by side so the source of System B's accuracy change is visible. Any cell that moves from the recycling-prediction column over to the garbage-prediction column reflects a contamination override that System B made.


In [ ]:
labels_order = ['recycling', 'garbage']

cm_a = confusion_matrix(test_df['ground_truth_pathway'], test_df['system_a_pred'], labels=labels_order)
cm_b = confusion_matrix(test_df['ground_truth_pathway'], test_df['system_b_pred'], labels=labels_order)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in zip(axes, [cm_a, cm_b],
                         ['System A (Stage A only)', 'System B (Stage A + Stage B)']):
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labels_order))); ax.set_yticks(range(len(labels_order)))
    ax.set_xticklabels(labels_order);        ax.set_yticklabels(labels_order)
    ax.set_xlabel('Predicted Pathway');      ax.set_ylabel('Ground Truth Pathway')
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=13)

plt.suptitle('Disposal Pathway Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(EVAL_RESULTS_PATH / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.4 McNemar's Contingency Table

McNemar's test operates on a 2x2 table of paired outcomes. We classify each test image into one of four cells based on whether each system got it right or wrong. The discordant cells `b` (System A right, System B wrong) and `c` (System A wrong, System B right) drive the test. If `c >> b`, System B is making systematic improvements over System A.


In [ ]:
a_right_b_right = int(( test_df['system_a_correct']  &  test_df['system_b_correct']).sum())
a_right_b_wrong = int(( test_df['system_a_correct']  & ~test_df['system_b_correct']).sum())
a_wrong_b_right = int((~test_df['system_a_correct'] &  test_df['system_b_correct']).sum())
a_wrong_b_wrong = int((~test_df['system_a_correct'] & ~test_df['system_b_correct']).sum())

contingency = np.array([
    [a_right_b_right, a_right_b_wrong],
    [a_wrong_b_right, a_wrong_b_wrong],
])

contingency_df = pd.DataFrame(
    contingency,
    index=['System A correct', 'System A wrong'],
    columns=['System B correct', 'System B wrong'],
)
contingency_df.to_csv(EVAL_RESULTS_PATH / 'contingency_table.csv')

print("Contingency Table")
print("=" * 50)
print(contingency_df.to_string())
print()
print("Discordant pairs (the cells McNemar's test acts on)")
print(f"  b -- System A right, System B wrong : {a_right_b_wrong}")
print(f"  c -- System A wrong, System B right : {a_wrong_b_right}")


### 6.5 Test Statistic and p-value

We run McNemar's test with continuity correction using `statsmodels`. The null hypothesis is that the two systems disagree symmetrically (System B is just as likely to make a unique error as System A). We reject the null at **alpha = 0.05** if the p-value falls below 0.05, meaning the asymmetry in discordant pairs is unlikely to be due to chance.


In [ ]:
result = mcnemar(contingency, exact=False, correction=True)
alpha  = 0.05

print("McNemar's Test (with continuity correction)")
print("=" * 50)
print(f"Test statistic (chi-squared) : {result.statistic:.4f}")
print(f"p-value                      : {result.pvalue:.6f}")
print(f"Significance level (alpha)   : {alpha}")
print()
if result.pvalue < alpha:
    print(f"REJECT the null hypothesis. System B differs significantly from System A "
          f"(p = {result.pvalue:.4f} < {alpha}).")
else:
    print(f"FAIL TO REJECT the null hypothesis. No significant difference between systems "
          f"(p = {result.pvalue:.4f} >= {alpha}).")


After reading the test statistic, we can confirm that adding the Stage B contamination classifier produces a **(fill in: statistically significant / not statistically significant)** improvement over Stage A alone (chi-squared = **XX.XX**, p = **0.XXXX**). The discordant cell asymmetry (`c = XX` System A wrong / System B right vs `b = XX` the reverse) is what drives this result.


### 6.6 Supplementary Override Metrics

McNemar's test answers the significance question but does not directly measure whether System B's overrides are *accurate*. We define an **override event** as any image where System A predicted recycling but System B predicted garbage. **Override precision** is the share of overrides that were truly contaminated recyclables. **Override recall** is the share of contaminated recyclables in the test set that System B successfully caught.


In [ ]:
override_mask = (test_df['system_a_pred'] == 'recycling') & (test_df['system_b_pred'] == 'garbage')

n_overrides          = int(override_mask.sum())
n_correct_overrides  = int((override_mask & test_df['is_contaminated_recyclable']).sum())
n_contaminated_total = int(test_df['is_contaminated_recyclable'].sum())
n_caught_by_system_b = int((test_df['is_contaminated_recyclable']
                            & (test_df['system_b_pred'] == 'garbage')).sum())

override_precision = n_correct_overrides / n_overrides           if n_overrides           > 0 else 0.0
override_recall    = n_caught_by_system_b / n_contaminated_total if n_contaminated_total > 0 else 0.0

print("Supplementary Override Metrics")
print("=" * 55)
print(f"Total overrides (A=recycling, B=garbage)   : {n_overrides}")
print(f"Correct overrides (true contaminated)      : {n_correct_overrides}")
print(f"Total contaminated recyclables in test set : {n_contaminated_total}")
print(f"Contaminated recyclables caught by B       : {n_caught_by_system_b}")
print()
print(f"Override Precision : {override_precision:.4f}")
print(f"Override Recall    : {override_recall:.4f}")

# Persist for the blog post / paper appendix
pd.DataFrame([{
    'overrides':            n_overrides,
    'correct_overrides':    n_correct_overrides,
    'contaminated_total':   n_contaminated_total,
    'caught_by_system_b':   n_caught_by_system_b,
    'override_precision':   override_precision,
    'override_recall':      override_recall,
}]).to_csv(EVAL_RESULTS_PATH / 'override_metrics.csv', index=False)
test_df.to_csv(EVAL_RESULTS_PATH / 'per_image_predictions.csv', index=False)
print(f"\nSaved override metrics + per-image predictions to {EVAL_RESULTS_PATH}")


## Package Outputs for Download

Bundle the trained model checkpoints, every results directory, and the Stage B manifest into a single zip in `/kaggle/working/`. After Run All completes, `ecobin_outputs.zip` appears in the Kaggle session's output panel and is downloadable in one click.


In [ ]:
import zipfile

OUTPUT_ZIP = WORKING_PATH / 'ecobin_outputs.zip'
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

# Roots that get walked recursively into the zip
roots = [
    (MODELS_PATH,          'Models'),
    (STAGE_A_RESULTS_PATH, 'Stage A Results'),
    (STAGE_B_RESULTS_PATH, 'Stage B Results'),
    (EVAL_RESULTS_PATH,    'McNemar Evaluation Results'),
    (DATASET_AUDIT_PATH,   'Waste Classification Dataset Audit'),
]

# Individual files we also want to include
extras = [
    DATASET_ROOT / 'stage_b_manifest.csv',
    DATASET_ROOT / 'dataset_audit.png',
    DATASET_ROOT / 'visual_sample_audit.png',
]

with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, label in roots:
        if not root.exists():
            continue
        for f in root.rglob('*'):
            if f.is_file():
                zf.write(f, arcname=f"{label}/{f.relative_to(root)}")
    for f in extras:
        if f.exists():
            zf.write(f, arcname=f"Synthetic Recyclable Contamination Dataset/{f.name}")

size_mb = OUTPUT_ZIP.stat().st_size / (1024 * 1024)
print(f"Wrote {OUTPUT_ZIP} ({size_mb:.1f} MB)")
print("\nContents:")
with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    for n in zf.namelist():
        print(f"  {n}")
